# Phase 4 — WaveLite-Final / ALL / RUN MỚI

Kiến trúc đích: **Haar DWT → Mamba/UNet trên LL, giữ dải cao → IWT → DEN một/hai U-block và một depthwise block/group
với ba nhánh FFT + attention → residual depthwise detail → RGB**.

Giữ năng lực khôi phục ngữ cảnh của WDMamba; giảm lặp ở DEN để tăng FPS; bổ sung detail adapter nhỏ.
Mục tiêu đo: **ΔPSNR ≥ −0,30 dB**, **FPS ≥ 2×**, FPS 256 ≥ 1,10× và FPS/MSE không giảm so tham chiếu.
Đây là điều kiện kiểm tra, chưa phải kết quả đạt được. Test không quyết định trọng số hoặc runtime.
Notebook chứa toàn bộ runner. O-HAZY và I-HAZE dùng bản có sẵn trong VDT_GD2;
RESIDE-6K và SOTS tự tải qua KaggleHub khi chưa có dữ liệu local.

**Chọn GPU rồi Run all, không cần notebook khôi phục hoặc kết quả cũ.**
Mỗi lần chạy cell cấu hình tạo một thư mục run mới. Dataset và checkpoint tải về được cache chung.
SOTS dùng checkpoint RESIDE-6K; O-HAZY tối ưu trọng số cùng miền; I-HAZE train hai model
khởi tạo ngẫu nhiên độc lập. Nguồn trọng số được ghi trong metadata.

Kiểm tra dataset/checkpoint trước, sau đó kiểm tra môi trường và forward GPU của cả bốn job.
Chỉ bắt đầu train sau khi cả bốn job sẵn sàng; lỗi chuẩn bị sẽ dừng với thông báo cụ thể.

## 1. Chọn dataset, đường dẫn và artifact

`PRESET` chọn lịch chạy: `quick` giảm thời gian train/benchmark, `standard` dùng lịch dài hơn.
| Thiết lập | quick | standard |
|---|---:|---:|
| O-HAZY, tối đa số bước | 900 | 4.000 |
| I-HAZE, mỗi model | 5.000 | 27.000 |
| Lặp đo tốc độ | 15 | 40 |

`quick` giữ một DEN U-block/group để ưu tiên FPS; `standard` giữ hai block đầu đã học từ
checkpoint RESIDE trước khi thêm adapter nhẹ, giúp ưu tiên PSNR và vẫn giảm đáng kể chi phí.

Cả hai đều tính chất lượng trên toàn bộ test. SOTS dùng checkpoint RESIDE-6K để khởi tạo model nhẹ,
phục hồi trên RESIDE-6K/train hoặc TRAIN_ROOT riêng. Các block giữ lại nhận trực tiếp trọng số
checkpoint; teacher loss tiếp tục giữ đầu ra Reference trong lúc phục hồi.

`DATASET_NAME='all'` chạy lần lượt **SOTS Indoor → SOTS Outdoor → O-HAZY → I-HAZE**;
không bao gồm RESIDE-6K riêng. Mỗi dataset có thư mục riêng, cuối lượt có bảng tổng hợp.

Preset `quick` ưu tiên thời gian: RESIDE/SOTS nạp chung checkpoint RESIDE-6K và phục hồi model nhẹ
trên dữ liệu train riêng, sau đó đánh giá đầy đủ benchmark. O-HAZY dùng lịch phục hồi ngắn.
I-HAZE vẫn khởi tạo ngẫu nhiên hai model độc lập để so sánh.
SOTS chuẩn là benchmark test. Không có `TRAIN_ROOT` riêng thì tự lấy phần train của RESIDE-6K;
không cập nhật trọng số trên SOTS test. `DATA_ROOT_OVERRIDE` và `CHECKPOINTS` cho phép dùng
các file đã có trên Drive hoặc runtime cũ. `AUTO_DOWNLOAD_DATASETS=True` tự tải RESIDE-6K/SOTS
khi thiếu, tái dùng cache cho các lượt sau. SOTS chỉ tải một lần cho Indoor/Outdoor.
`RUN_MODE='auto'` tiếp tục lượt dở hoặc benchmark lượt đã xong.

Bản chạy mới: `quick` I-HAZE dùng đủ 4.000 bước crop256 + 1.000 bước ảnh đầy đủ
cho mỗi model, chọn checkpoint tốt nhất bằng validation. Không dừng ở 1.700 bước như lượt cũ.
`standard` vẫn có early stopping. SOTS tự dùng RESIDE-6K/train nếu chưa có TRAIN_ROOT riêng,
phục hồi kiến trúc nhẹ tối đa 1.800 + 600 bước ở quick, chọn bằng validation tách theo scene.
Giới hạn validation quick là 32 scene/64 cặp; test vẫn đủ 500 ảnh mỗi miền.
Để các đường dẫn checkpoint trống nếu muốn tự tải archive; chỉ điền khi đã có file local.

In [ ]:
DATA_DOWNLOAD_SOURCE = '"""Resolve existing datasets or download the external synthetic benchmarks."""\nfrom pathlib import Path\nimport json\n\n\ndef pair_root(root):\n    root = Path(root)\n    dirs = {p.name.lower(): p for p in root.iterdir() if p.is_dir()} if root.is_dir() else {}\n    clear = next((dirs[k] for k in (\'clear\', \'gt\', \'clean\', \'groundtruth\', \'ground_truth\') if k in dirs), None)\n    hazy = next((dirs[k] for k in (\'hazy\', \'haze\', \'input\', \'inputs\', \'hazy_images\') if k in dirs), None)\n    return clear is not None and hazy is not None and any(p.is_file() for p in clear.iterdir()) and any(p.is_file() for p in hazy.iterdir())\n\n\ndef find_dataset(base, name, test_only=False):\n    base = Path(base).expanduser().resolve()\n    candidates = [base, *sorted((p for p in base.rglob(\'*\') if p.is_dir()), key=lambda p: (len(p.parts), str(p)))]\n    if name.startswith(\'sots-\'):\n        domain = \'indoor\' if name == \'sots-its\' else \'outdoor\'\n        candidates = [p for p in candidates if domain in {x.lower() for x in p.parts}]\n    if name == \'reside6k\':\n        if not test_only:\n            for p in candidates:\n                children = {d.name.lower() for d in p.iterdir() if d.is_dir()}\n                if {\'train\', \'test\'} <= children:\n                    return p\n        tests = [p for p in candidates if \'test\' in {x.lower() for x in p.parts} and pair_root(p)]\n        if tests:\n            return tests[0]\n        # Never silently evaluate a training directory when the test layout is missing.\n        raise FileNotFoundError(f\'Không tìm thấy RESIDE-6K test/clear và test/hazy dưới {base}.\')\n    for p in candidates:\n        if pair_root(p):\n            return p\n    raise FileNotFoundError(f\'Không tìm thấy cặp clear/hazy cho {name} dưới {base}.\')\n\n\ndef resolve_datasets(names, specs, project_root, overrides, cache_root, *,\n                     allow_download=True, test_only=False, download=None,\n                     reside_handle=\'kmljts/reside-6k\',\n                     sots_handle=\'balraj98/synthetic-objective-testing-set-sots-reside\',\n                     continue_on_error=False):\n    """Return actual roots plus errors; an explicit local override never triggers a download."""\n    project_root, cache_root = Path(project_root), Path(cache_root)\n    cache_root.mkdir(parents=True, exist_ok=True)\n    manifest = cache_root/\'downloads.json\'\n    cached = json.loads(manifest.read_text(encoding=\'utf-8\')) if manifest.is_file() else {}\n    roots, errors, downloads = {}, {}, {}\n    for name in names:\n        try:\n            explicit = overrides.get(name)\n            local = Path(explicit).expanduser().resolve() if explicit else project_root/specs[name][\'relative_root\']\n            if explicit:\n                if not local.is_dir():\n                    raise FileNotFoundError(f\'Đường dẫn local của {name} không tồn tại: {local}\')\n                roots[name] = str(local)\n                continue\n            if name in {\'o-hazy\', \'i-haze\'}:\n                if not local.is_dir():\n                    raise FileNotFoundError(f\'{name} phải lấy từ checkout VDT_GD2: {local}\')\n                roots[name] = str(local.resolve())\n                continue\n            handle = reside_handle if name == \'reside6k\' else sots_handle\n            if local.is_dir():\n                try:\n                    roots[name] = str(find_dataset(local, name, test_only))\n                    continue\n                except FileNotFoundError:\n                    pass\n            if handle in cached and Path(cached[handle]).is_dir():\n                try:\n                    roots[name] = str(find_dataset(cached[handle], name, test_only))\n                    continue\n                except FileNotFoundError:\n                    pass\n            if not allow_download:\n                raise FileNotFoundError(f\'Thiếu {name}; bật AUTO_DOWNLOAD_DATASETS hoặc đặt DATA_ROOT_OVERRIDE(S).\')\n            if handle not in downloads:\n                print(f\'Đang tải/tái dùng cache KaggleHub: {handle}\', flush=True)\n                if download is None:\n                    import kagglehub\n                    downloads[handle] = Path(kagglehub.dataset_download(handle))\n                else:\n                    downloads[handle] = Path(download(handle))\n                cached[handle] = str(downloads[handle].resolve())\n                manifest.write_text(json.dumps(cached, indent=2), encoding=\'utf-8\')\n            roots[name] = str(find_dataset(downloads[handle], name, test_only))\n        except Exception as exc:\n            if not continue_on_error:\n                raise\n            errors[name] = str(exc)\n            print(f\'{name}: chuẩn bị dataset lỗi: {exc}\', flush=True)\n    return roots, errors\n\n\ndef resolve_training_roots(names, specs, project_root, train_roots, overrides, cache_root,\n                           *, enabled=True, allow_download=True, download=None,\n                           reside_handle=\'kmljts/reside-6k\'):\n    """Resolve separate training data; never substitute a SOTS benchmark directory."""\n    roots = dict(train_roots)\n    missing = [n for n in names if n.startswith(\'sots-\') and not roots.get(n)]\n    if not enabled or not missing:\n        return roots\n    resolved, _ = resolve_datasets([\'reside6k\'], specs, project_root, overrides, cache_root,\n        allow_download=allow_download, test_only=False, download=download,\n        reside_handle=reside_handle)\n    base = Path(resolved[\'reside6k\'])\n    train = next((p for p in base.iterdir() if p.is_dir() and p.name.lower() == \'train\'), None)\n    if train is None or not pair_root(train):\n        raise FileNotFoundError(f\'Cần RESIDE-6K/train với các cặp GT/hazy, không dùng test: {base}\')\n    for name in missing:\n        roots[name] = str(train.resolve())\n    return roots\n'
DATASET_SPECS = {'o-hazy': {'label': 'O-HAZY', 'relative_root': 'dataset/O-HAZY', 'kind': 'paired', 'can_train': True, 'val_ids': [5, 15, 25, 35]}, 'i-haze': {'label': 'I-HAZE', 'relative_root': 'dataset/I-HAZE', 'kind': 'paired', 'can_train': True, 'val_ids': []}, 'reside6k': {'label': 'RESIDE-6K', 'relative_root': 'dataset/RESIDE-6K', 'kind': 'paired', 'can_train': True, 'val_ids': []}, 'sots-its': {'label': 'SOTS Indoor / ITS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/indoor', 'kind': 'evaluation-only', 'can_train': False, 'val_ids': []}, 'sots-ots': {'label': 'SOTS Outdoor / OTS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/outdoor', 'kind': 'evaluation-only', 'can_train': False, 'val_ids': []}}

from pathlib import Path
import ast, json, os, platform, subprocess, sys

DATASET_NAME = 'all'  # @param ['all','o-hazy','sots-its','sots-ots','i-haze','reside6k']
PRESET = 'standard'     # @param ['quick','standard']
LOCAL_PROJECT = ''
ARCH_ROOT_OVERRIDE = ''  # checkout WDMamba đã có; để trống sẽ tự clone
ARCH_REPO_URL = 'https://github.com/Darkin72/Haze-Clear.git'
DATA_ROOT_OVERRIDE = ''
DATA_ROOT_OVERRIDES = {'sots-its':'', 'sots-ots':'', 'o-hazy':'', 'i-haze':'', 'reside6k':''}  # dùng khi all
AUTO_DOWNLOAD_DATASETS = True  # tải RESIDE-6K/SOTS khi chưa có đường dẫn local
KAGGLE_RESIDE6K_DATASET = 'kmljts/reside-6k'
KAGGLE_SOTS_DATASET = 'balraj98/synthetic-objective-testing-set-sots-reside'
DATA_DOWNLOAD_ROOT = '/content/wdmamba_phase4/datasets_cache'
TRAIN_ROOT = ''             # ITS/OTS train riêng, tách khỏi benchmark SOTS
TRAIN_ROOTS = {'sots-its':'', 'sots-ots':''}  # ưu tiên train riêng nếu đã có
AUTO_RESIDE_TRAIN = True    # tự tải/tái dùng RESIDE-6K/train khi chưa điền TRAIN_ROOT(S)
REQUIRE_COMPRESSION = True  # dừng nếu chỉ có bản sao Reference, không gọi nó là model cải tiến
EVAL_ONLY = None            # None tự nhận diện; False bắt buộc train/val
CHECKPOINTS = {'o-hazy': '', 'sots-its': '', 'sots-ots': '', 'reside6k': ''}
RESIDE_CHECKPOINT = ''      # Điền một lần: dùng chung cho RESIDE-6K, SOTS Indoor và Outdoor
STUDENT_CHECKPOINT = ''     # tùy chọn: Phase 3 O-HAZY hoặc student cùng dataset
CHECKPOINT_ZIP = ''         # ZIP trọng số có sẵn, không phải dataset ZIP
CHECKPOINT_SEARCH_ROOTS = [
    '/content/wdmamba_o_hazy_final/weights', '/content/wdmamba_v2',
    '/content/wdmamba_sots', '/content/WDMamba_ckpts',
]
ALLOW_WEIGHT_DOWNLOAD = True   # tải một lần bộ checkpoint dùng chung qua Google Drive
from datetime import datetime, timezone
from uuid import uuid4
RUN_NAME = 'wavelite_all_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:6]
SHARED_WEIGHT_ROOT = Path('/content/wdmamba_phase4/shared_checkpoints')
RUN_MODE = 'auto'           # auto | train | resume | benchmark
EVAL_MAX_SIDE = 768         # 0 = kích thước gốc; giữ 768 khi đối chiếu các phase cũ
PYTHON_OVERRIDE = ''        # Python CUDA/Mamba đã có, nếu cần

if platform.system() != 'Linux':
    raise RuntimeError('Notebook cần Colab/Linux GPU; kiểm thử CPU dùng test_cpu.py.')
PROJECT_ROOT = Path(LOCAL_PROJECT).expanduser().resolve() if LOCAL_PROJECT else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p/'dataset').is_dir()), Path('/content/VDT_GD2'))
ACTIVE_DATASETS = ['sots-its','sots-ots','o-hazy','i-haze'] if DATASET_NAME == 'all' else [DATASET_NAME]
if DATASET_NAME == 'all' and (DATA_ROOT_OVERRIDE or TRAIN_ROOT):
    raise ValueError('Khi all, điền DATA_ROOT_OVERRIDES / TRAIN_ROOTS cho từng dataset.')
SPEC = DATASET_SPECS[ACTIVE_DATASETS[0]]
RUN_ROOT = Path('/content/wdmamba_phase4_new') / DATASET_NAME / RUN_NAME
OUTPUT_ROOT = RUN_ROOT/'results'
RUN_ROOT.mkdir(parents=True, exist_ok=True); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def command(args, cwd=None, log='setup.log'):
    args = list(map(str, args)); print('$', ' '.join(args), flush=True)
    with (RUN_ROOT/log).open('a', encoding='utf-8') as out:
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True); out.write(line); out.flush()
                rc = proc.wait()
            except BaseException:
                proc.kill(); proc.wait(); raise
    if rc: raise subprocess.CalledProcessError(rc, args)

if not PROJECT_ROOT.exists():
    command(['git','clone','--depth','1','--filter=blob:none','--sparse',
             'https://github.com/Darkin72/VDT_GD2.git',PROJECT_ROOT])
    command(['git','sparse-checkout','set','--skip-checks',
             *[DATASET_SPECS[name]['relative_root'] for name in ACTIVE_DATASETS if name in {'o-hazy','i-haze'}],
             'solution/Haze Clear'], cwd=PROJECT_ROOT)
DATA_ROOT = Path(DATA_ROOT_OVERRIDE).expanduser().resolve() if DATA_ROOT_OVERRIDE else PROJECT_ROOT/SPEC['relative_root']
if (PROJECT_ROOT/'.git/info/sparse-checkout').is_file():
    # Chỉ O-HAZY/I-HAZE nằm trong VDT_GD2; RESIDE-6K và SOTS lấy từ KaggleHub.
    missing = [DATASET_SPECS[name]['relative_root'] for name in ACTIVE_DATASETS
               if name in {'o-hazy', 'i-haze'}
               if not (PROJECT_ROOT/DATASET_SPECS[name]['relative_root']).is_dir()
               and not (DATA_ROOT_OVERRIDES.get(name) if DATASET_NAME == 'all' else DATA_ROOT_OVERRIDE)]
    if missing: command(['git','sparse-checkout','add','--skip-checks',*missing], cwd=PROJECT_ROOT)

dataset_loader = {'__name__': 'phase4_download_bootstrap'}
exec(compile(DATA_DOWNLOAD_SOURCE, 'phase4_download.py', 'exec'), dataset_loader)

def download_dataset(handle):
    try:
        import kagglehub
    except ImportError:
        command([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], log='dataset_download.log')
        import kagglehub
    return kagglehub.dataset_download(handle)

data_overrides = dict(DATA_ROOT_OVERRIDES) if DATASET_NAME == 'all' else {DATASET_NAME: DATA_ROOT_OVERRIDE}
RESOLVED_DATA_ROOTS, DATASET_ERRORS = dataset_loader['resolve_datasets'](
    ACTIVE_DATASETS, DATASET_SPECS, PROJECT_ROOT, data_overrides, DATA_DOWNLOAD_ROOT,
    allow_download=AUTO_DOWNLOAD_DATASETS, test_only=False, download=download_dataset,
    reside_handle=KAGGLE_RESIDE6K_DATASET, sots_handle=KAGGLE_SOTS_DATASET,
    continue_on_error=False)
if DATASET_NAME == 'all':
    TRAIN_ROOTS = dataset_loader['resolve_training_roots'](
        ACTIVE_DATASETS, DATASET_SPECS, PROJECT_ROOT, TRAIN_ROOTS, data_overrides,
        DATA_DOWNLOAD_ROOT, enabled=AUTO_RESIDE_TRAIN, allow_download=AUTO_DOWNLOAD_DATASETS,
        download=download_dataset, reside_handle=KAGGLE_RESIDE6K_DATASET)
elif DATASET_NAME.startswith('sots-') and not TRAIN_ROOT:
    TRAIN_ROOT = dataset_loader['resolve_training_roots'](
        [DATASET_NAME], DATASET_SPECS, PROJECT_ROOT, {DATASET_NAME: ''}, data_overrides,
        DATA_DOWNLOAD_ROOT, enabled=AUTO_RESIDE_TRAIN, allow_download=AUTO_DOWNLOAD_DATASETS,
        download=download_dataset, reside_handle=KAGGLE_RESIDE6K_DATASET).get(DATASET_NAME, '')
DATA_ROOT = Path(RESOLVED_DATA_ROOTS.get(ACTIVE_DATASETS[0], PROJECT_ROOT/SPEC['relative_root']))
if DATASET_NAME == 'all':
    DATA_ROOT_OVERRIDES.update(RESOLVED_DATA_ROOTS)
LOCAL_ARCH = Path(ARCH_ROOT_OVERRIDE).expanduser().resolve() if ARCH_ROOT_OVERRIDE else PROJECT_ROOT/'solution/Haze Clear'
REPO_ROOT = LOCAL_ARCH
if not (REPO_ROOT/'basicsr/archs/wavemamba_arch.py').is_file():
    # VDT_GD2 chứa dataset; WDMamba là repo mã nguồn độc lập để notebook clone được
    # trực tiếp và không phụ thuộc sparse-checkout của repo cha.
    REPO_ROOT = Path('/content/WDMamba')
    if not (REPO_ROOT/'basicsr/archs/wavemamba_arch.py').is_file():
        command(['git','clone','--depth','1',ARCH_REPO_URL,REPO_ROOT])
if not (REPO_ROOT/'basicsr/archs/wavemamba_arch.py').is_file():
    raise FileNotFoundError(f'Không tìm thấy WDMamba architecture tại {REPO_ROOT}.')

CONFIG = {
    'dataset_name': ACTIVE_DATASETS[0], 'project_root': str(PROJECT_ROOT), 'repo_root': str(REPO_ROOT),
    'data_root': str(DATA_ROOT), 'train_root': TRAIN_ROOT, 'eval_only': EVAL_ONLY,
    'work_root': str(RUN_ROOT), 'output_root': str(OUTPUT_ROOT),
    'checkpoint': CHECKPOINTS.get(DATASET_NAME, ''), 'reside_checkpoint': RESIDE_CHECKPOINT,
    'student_checkpoint': STUDENT_CHECKPOINT,
    'checkpoint_zip': CHECKPOINT_ZIP, 'checkpoint_search_roots': CHECKPOINT_SEARCH_ROOTS,
    'allow_weight_download': ALLOW_WEIGHT_DOWNLOAD,
    'require_compression': REQUIRE_COMPRESSION,
    'seed': 2040, 'device': 'cuda', 'den_blocks': 1, 'detail_width': 16, 'reference_depth': 4,
    'train_max_side': EVAL_MAX_SIDE, 'eval_max_side': EVAL_MAX_SIDE, 'crop_border': 4,
    'cache_images': 40, 'warmup': 8, 'timing_repeats': 40, 'log_every': 50, 'val_every': 200,
    'patience': 8, 'l1_weight': 0.02, 'edge_weight': 0.01, 'kd_weight': 0.10,
    'ema_decay': 0.999, 'weight_decay': 1e-5,
    'psnr_tolerance_db': 0.30, 'target_speedup': 2.0, 'target_speedup_256': 1.10,
    'deploy_psnr_tolerance_db': 0.01, 'fusion_max_error': 1e-4,
    'adapt_schedule': [
        {'steps': 3000, 'crop': 384, 'accum': 2, 'lr': 1e-4, 'den_only': True},
        {'steps': 1000, 'crop': 0, 'accum': 1, 'lr': 1e-5, 'trunk_lr': 2e-6}],
    'scratch_schedule': [
        {'steps': 20000, 'crop': 256, 'accum': 2, 'lr': 2e-4},
        {'steps': 5000, 'crop': 384, 'accum': 2, 'lr': 5e-5},
        {'steps': 2000, 'crop': 0, 'accum': 1, 'lr': 1e-5}],
    'reference_schedule': [
        {'steps': 20000, 'crop': 256, 'accum': 2, 'lr': 2e-4},
        {'steps': 5000, 'crop': 384, 'accum': 2, 'lr': 5e-5},
        {'steps': 2000, 'crop': 0, 'accum': 1, 'lr': 1e-5}],
}
if PRESET not in ('quick', 'standard'): raise ValueError('PRESET không hợp lệ.')
CONFIG['preset'] = PRESET
if PRESET == 'quick':
    CONFIG.update(runtime_only_datasets=[],
                  cache_shape_benchmarks=True, skip_duplicate_eager=True,
                  warmup=5, timing_repeats=15, val_every=100, patience=4)
    CONFIG['adapt_schedule'] = [
        {'steps':600, 'crop':384, 'accum':1, 'lr':2e-5, 'den_only':True},
        {'steps':300, 'crop':0, 'accum':1, 'lr':8e-6, 'trunk_lr':1e-6}]
    # I-HAZE is the only branch that still needs random initialization.
    CONFIG['scratch_schedule'] = [
        {'steps':4000, 'crop':256, 'accum':1, 'lr':2e-4},
        {'steps':1000, 'crop':0, 'accum':1, 'lr':2e-5}]
    CONFIG['reference_schedule'] = list(CONFIG['scratch_schedule'])
    CONFIG['synthetic_adapt_schedule'] = [
        {'steps':1800, 'crop':256, 'accum':1, 'lr':1e-4, 'den_only':True},
        {'steps':600, 'crop':256, 'accum':1, 'lr':2e-5, 'trunk_lr':2e-6}]
    CONFIG['synthetic_val_max_scenes'] = 32
    CONFIG['synthetic_val_max_pairs'] = 64
    CONFIG['synthetic_val_every'] = 200
    CONFIG['synthetic_patience'] = 12
else:
    # Standard keeps two learned DEN blocks from the RESIDE teacher before the
    # lightweight adapter, preserving more checkpoint capacity for PSNR.
    CONFIG['den_blocks'] = 2
if DATASET_NAME in ('reside6k', 'sots-its', 'sots-ots'):
    CONFIG['adapt_schedule'] = CONFIG.get('synthetic_adapt_schedule', CONFIG['adapt_schedule'])
    CONFIG['val_max_scenes'] = CONFIG.get('synthetic_val_max_scenes', 0)
    CONFIG['val_max_pairs'] = CONFIG.get('synthetic_val_max_pairs', 0)
    CONFIG['val_every'] = CONFIG.get('synthetic_val_every', CONFIG['val_every'])
    CONFIG['patience'] = CONFIG.get('synthetic_patience', CONFIG['patience'])
    CONFIG['exclude_benchmark_overlap'] = DATASET_NAME.startswith('sots-')
CONFIG_PATH = RUN_ROOT/'config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


## 2. Checkpoint dùng chung

Tự dò file local hoặc tải một archive cho SOTS/O-HAZY. I-HAZE không cần checkpoint.
Nếu Drive lỗi/quota, dừng ở đây và điền `CHECKPOINT_ZIP` local; chưa có model nào được train.

In [ ]:
MODULE_SOURCES = {'phase4_arch.py': '"""WaveLite-Final: wavelet/Mamba trunk, compact DEN and residual detail."""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport copy\nimport importlib.machinery\nimport importlib.util\nimport re\nimport sys\nimport torch\nfrom torch import nn\nimport phase4_data as data\n\nARCH_VERSION = \'p4-wavelite-uden1or2-dw-detail-v2\'\n\n\n@lru_cache(maxsize=1)\ndef import_architecture(root):\n    for name, rel in [(\'basicsr\', \'basicsr\'), (\'basicsr.utils\', \'basicsr/utils\'),\n                      (\'basicsr.archs\', \'basicsr/archs\')]:\n        if name in sys.modules:\n            raise RuntimeError(f\'Cần process sạch; đã import {name}.\')\n        spec = importlib.machinery.ModuleSpec(name, loader=None, is_package=True)\n        spec.submodule_search_locations = [str(Path(root) / rel)]\n        sys.modules[name] = importlib.util.module_from_spec(spec)\n    from basicsr.archs.wavemamba_arch import WaveMamba\n    from basicsr.archs.detail_enhance_net import DENet\n    return WaveMamba, DENet\n\n\ndef checkpoint_state(path):\n    obj = torch.load(path, map_location=\'cpu\', weights_only=False)\n    for key in (\'params_ema\', \'params\', \'state_dict\', \'model\'):\n        if isinstance(obj, dict) and isinstance(obj.get(key), dict):\n            obj = obj[key]\n            break\n    if not isinstance(obj, dict) or not obj or not all(torch.is_tensor(v) for v in obj.values()):\n        raise ValueError(f\'Không có state_dict WDMamba hợp lệ: {path}\')\n    return {k.removeprefix(\'module.\'): v for k, v in obj.items()}\n\n\ndef detect_den_depth(state):\n    groups = {i: set() for i in (1, 2, 3)}\n    for key in state:\n        m = re.fullmatch(r\'restoration_network\\.DE\\.g([123])\\.gp\\.(\\d+)\\.conv1\\.weight\', key)\n        if m:\n            groups[int(m[1])].add(int(m[2]))\n    depths = {len(v) for v in groups.values()}\n    if len(depths) != 1 or next(iter(depths)) not in (1, 2, 3, 4, 6):\n        raise ValueError(f\'Không nhận diện được độ sâu DEN: {groups}\')\n    depth = next(iter(depths))\n    if any(v != set(range(depth)) for v in groups.values()):\n        raise ValueError(\'Chỉ số block DEN không liên tục.\')\n    return depth\n\n\nclass FeatureAdapter(nn.Module):\n    """A second, cheap DEN block; exact identity at initialization."""\n    def __init__(self, channels=16):\n        super().__init__()\n        self.body = nn.Sequential(nn.Conv2d(channels, 2*channels, 1), nn.SiLU(),\n                                  nn.Conv2d(2*channels, 2*channels, 5, padding=2, groups=2*channels),\n                                  nn.SiLU(), nn.Conv2d(2*channels, channels, 1))\n        nn.init.zeros_(self.body[-1].weight)\n        nn.init.zeros_(self.body[-1].bias)\n\n    def forward(self, x):\n        return x + self.body(x)\n\n\nclass DetailAdapter(nn.Module):\n    def __init__(self, width=16):\n        super().__init__()\n        self.body = nn.Sequential(nn.Conv2d(3, width, 1), nn.SiLU(),\n                                  nn.Conv2d(width, width, 3, padding=1, groups=width), nn.SiLU(),\n                                  nn.Conv2d(width, 3, 1))\n        nn.init.zeros_(self.body[-1].weight)\n        nn.init.zeros_(self.body[-1].bias)\n\n    def forward(self, x):\n        return self.body(x)\n\n\nclass PaddedModel(nn.Module):\n    """All measured forwards include pad/crop; no change to image resolution."""\n    def __init__(self, trunk, spec, adapter=None):\n        super().__init__()\n        self.trunk, self.adapter, self.spec = trunk, adapter, spec\n\n    def forward(self, x):\n        h, w = x.shape[-2:]\n        y = self.trunk(data.pad32(x))\n        if isinstance(y, (tuple, list)):\n            y = y[-1]\n        if self.adapter is not None:\n            y = y + self.adapter(y)\n        return y[..., :h, :w]\n\n\ndef new_trunk(repo_root, depth):\n    WaveMamba, DENet = import_architecture(str(Path(repo_root).resolve()))\n    trunk = WaveMamba(in_chn=3, wf=16, n_l_blocks=[1, 2, 2, 4], ffn_scale=2.)\n    trunk.restoration_network.DE = DENet(3, depth)\n    return trunk\n\n\ndef reference_model(repo_root, path=None, depth=4):\n    state = checkpoint_state(path) if path else None\n    if state is not None:\n        depth = detect_den_depth(state)\n    trunk = new_trunk(repo_root, depth)\n    if state is not None:\n        trunk.load_state_dict(state, strict=True)\n    return PaddedModel(trunk, {\'kind\': \'reference\', \'depth\': depth, \'detail_width\': 0})\n\n\ndef final_from_reference(reference, keep=1, detail_width=16):\n    if keep not in (1, 2):\n        raise ValueError(\'Phase 4 chỉ hỗ trợ giữ một hoặc hai U-block/group.\')\n    trunk = copy.deepcopy(reference.trunk)\n    depth = reference.spec[\'depth\']\n    if depth < keep:\n        raise ValueError(f\'Checkpoint có {depth} block/group, không đủ để giữ {keep} block đã học.\')\n    selected = list(range(keep))\n    for name in (\'g1\', \'g2\', \'g3\'):\n        group = getattr(trunk.restoration_network.DE, name)\n        gp = group.gp\n        channels = gp[depth].in_channels\n        kept = [copy.deepcopy(gp[i]) for i in range(keep)]\n        group.gp = nn.Sequential(*kept, FeatureAdapter(channels), copy.deepcopy(gp[depth]))\n    return PaddedModel(trunk, {\'kind\': \'final\', \'depth\': keep, \'feature_adapter\': True, \'detail_width\': detail_width},\n                       DetailAdapter(detail_width)), selected\n\n\ndef skeleton(repo_root, spec):\n    if spec[\'kind\'] == \'reference\':\n        return reference_model(repo_root, depth=spec[\'depth\'])\n    if spec[\'kind\'] != \'final\' or spec[\'depth\'] not in (1, 2) or not spec.get(\'feature_adapter\'):\n        raise ValueError(f\'Model spec không hợp lệ: {spec}\')\n    trunk = new_trunk(repo_root, spec[\'depth\'])\n    for name in (\'g1\', \'g2\', \'g3\'):\n        group = getattr(trunk.restoration_network.DE, name)\n        terminal = group.gp[-1]\n        kept = list(group.gp[:spec[\'depth\']])\n        group.gp = nn.Sequential(*kept, FeatureAdapter(terminal.in_channels), terminal)\n    return PaddedModel(trunk, dict(spec), DetailAdapter(spec[\'detail_width\']))\n\n\ndef transfer_summary(model):\n    """Parameter accounting immediately after transplant, before adaptation."""\n    total = sum(p.numel() for p in model.parameters())\n    new = sum(p.numel() for module in model.modules()\n              if isinstance(module, (FeatureAdapter, DetailAdapter)) for p in module.parameters())\n    return {\'inherited_params\': total-new, \'new_adapter_params\': new,\n            \'inherited_fraction\': (total-new)/max(1, total)}\n\n\n@torch.inference_mode(False)\n@torch.no_grad()\ndef fuse_conv_bn(model):\n    count = 0\n    for module in list(model.modules()):\n        if isinstance(module, nn.Sequential):\n            names = list(module._modules)\n            for left, right in zip(names, names[1:]):\n                conv, bn = module._modules[left], module._modules[right]\n                if isinstance(conv, nn.Conv2d) and isinstance(bn, nn.BatchNorm2d):\n                    module._modules[left] = torch.nn.utils.fusion.fuse_conv_bn_eval(conv, bn)\n                    module._modules[right] = nn.Identity()\n                    count += 1\n    return count\n', 'phase4_batch.py': '"""Standard-library orchestration for the notebook\'s all-dataset option."""\nfrom pathlib import Path\nimport copy\nimport csv\nimport html\nimport json\n\nALL_DATASETS = (\'sots-its\', \'sots-ots\', \'o-hazy\', \'i-haze\')\n\n\ndef make_configs(base, specs, root, data_roots=None, train_roots=None, checkpoints=None,\n                 students=None):\n    root = Path(root)\n    configs = []\n    for name in ALL_DATASETS:\n        cfg = copy.deepcopy(base)\n        work = root/\'datasets\'/name\n        cfg.update(dataset_name=name, work_root=str(work), output_root=str(work/\'results\'),\n                   data_root=str((data_roots or {}).get(name) or\n                                 Path(base[\'project_root\'])/specs[name][\'relative_root\']),\n                   train_root=(train_roots or {}).get(name, \'\'),\n                   checkpoint=(checkpoints or {}).get(name, \'\'),\n                   student_checkpoint=(students or {}).get(name, \'\'))\n        if name == \'i-haze\':\n            cfg[\'checkpoint\'] = cfg[\'student_checkpoint\'] = \'\'\n            cfg[\'eval_only\'] = False\n        else:\n            cfg[\'eval_only\'] = base.get(\'eval_only\')\n        if name.startswith(\'sots-\'):\n            cfg[\'adapt_schedule\'] = copy.deepcopy(base.get(\'synthetic_adapt_schedule\', base.get(\'adapt_schedule\', [])))\n            cfg[\'val_max_scenes\'] = base.get(\'synthetic_val_max_scenes\', 0)\n            cfg[\'val_max_pairs\'] = base.get(\'synthetic_val_max_pairs\', 0)\n            cfg[\'val_every\'] = base.get(\'synthetic_val_every\', base.get(\'val_every\', 200))\n            cfg[\'patience\'] = base.get(\'synthetic_patience\', base.get(\'patience\', 8))\n            cfg[\'exclude_benchmark_overlap\'] = True\n        configs.append(cfg)\n    return configs\n\n\ndef save_status(output, states):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    (output/\'suite_status.json\').write_text(json.dumps(states, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef prepare_jobs(configs, invoke, output):\n    states = []\n    for cfg in configs:\n        state = {\'dataset\': cfg[\'dataset_name\'], \'status\': \'ready\', \'error\': \'\'}\n        try:\n            invoke(cfg, \'prepare\')\n        except Exception as exc:\n            state.update(status=\'prepare_failed\', error=str(exc))\n            print(f\'{cfg["dataset_name"]}: chuẩn bị lỗi: {exc}\', flush=True)\n        states.append(state); save_status(output, states)\n    return states\n\n\ndef run_jobs(configs, states, invoke, mode, output):\n    for cfg, state in zip(configs, states):\n        if state[\'status\'] != \'ready\': continue\n        try:\n            invoke(cfg, mode)\n            state[\'status\'] = \'completed\'\n        except Exception as exc:\n            state.update(status=\'run_failed\', error=str(exc))\n            print(f\'{cfg["dataset_name"]}: chạy lỗi: {exc}\', flush=True)\n        save_status(output, states)\n        collect_reports(configs, states, output)\n    return collect_reports(configs, states, output)\n\n\ndef collect_reports(configs, states, output):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    summaries, final_rows, statuses = [], [], []\n    for cfg, state in zip(configs, states):\n        name, directory = cfg[\'dataset_name\'], Path(cfg[\'output_root\'])\n        if state[\'status\'] != \'completed\':\n            statuses.append(f\'| {name} | {state["status"]} |\')\n            continue\n        with (directory/\'summary.csv\').open(encoding=\'utf-8\') as f:\n            rows = list(csv.DictReader(f))\n        meta = json.loads((directory/\'metadata.json\').read_text(encoding=\'utf-8\'))\n        summaries.extend(rows)\n        if rows:\n            final_rows.append((name, rows[-1], meta[\'status\'], directory))\n        statuses.append(f\'| {name} | {meta["status"]} |\')\n    keys = list(dict.fromkeys(k for row in summaries for k in row)) or [\n        \'dataset\', \'model\', \'PSNR\', \'SSIM\', \'MSE\', \'FPS\', \'FPS_over_MSE\', \'GFLOPs_256_estimated_total\']\n    with (output/\'summary_all.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=keys); writer.writeheader(); writer.writerows(summaries)\n    lines = [\'# Phase 4 — Tổng hợp bốn bộ dữ liệu\', \'\',\n             \'| Dataset | PSNR | SSIM | FPS | FPS/MSE | GFLOPs 256* | Trạng thái |\',\n             \'|---|---:|---:|---:|---:|---:|---|\']\n    for name, row, status, _ in final_rows:\n        lines.append(f\'| {name} | {float(row["PSNR"]):.3f} | {float(row["SSIM"]):.4f} | \'\n                     f\'{float(row["FPS"]):.1f} | {float(row["FPS_over_MSE"]):,.0f} | \'\n                     f\'{float(row["GFLOPs_256_estimated_total"]):.2f} | {status} |\')\n    lines += [\'\', \'*GFLOPs gồm các thành phần ước tính; xem protocol từng dataset. \'\n              \'Không gộp trung bình PSNR/FPS giữa các dataset.\', \'\',\n              \'| Dataset | Tình trạng lượt chạy |\', \'|---|---|\', *statuses,\n              \'\', \'Lỗi chi tiết: suite_status.json. Mỗi dataset có report và metadata riêng.\']\n    (output/\'final_report_all.md\').write_text(\'\\n\'.join(lines)+\'\\n\', encoding=\'utf-8\')\n    escape = lambda s: html.escape(str(s))\n    table = \'\'.join(\'<tr>\'+\'\'.join(f\'<td>{escape(v)}</td>\' for v in (\n        name, f\'{float(row["PSNR"]):.3f}\', f\'{float(row["SSIM"]):.4f}\', f\'{float(row["FPS"]):.1f}\',\n        f\'{float(row["FPS_over_MSE"]):,.0f}\', f\'{float(row["GFLOPs_256_estimated_total"]):.2f}\', status))+\'</tr>\'\n        for name,row,status,_ in final_rows)\n    document = \'\'\'<!doctype html><html lang="vi"><meta charset="utf-8"><title>Phase 4 — All</title>\n<style>body{background:#f3f6fb;color:#162238;font:16px/1.6 system-ui;margin:48px auto;max-width:1100px;padding:24px}\ntable{width:100%;border-collapse:collapse;background:white}td,th{padding:18px;text-align:right;border-bottom:1px solid #e8edf4}td:first-child,th:first-child{text-align:left}small{color:#61718c}</style>\n<small>KẾT QUẢ THỰC NGHIỆM · PHASE 4</small><h1>Bốn bộ dữ liệu</h1>\n<table><tr><th>Dataset</th><th>PSNR</th><th>SSIM</th><th>FPS</th><th>FPS/MSE</th><th>GFLOPs 256*</th><th>Trạng thái</th></tr>\'\'\'\n    document += table+\'</table><p>*GFLOPs có thành phần ước tính. Không lấy trung bình giữa các dataset.</p>\'\n    document += \'<ul>\'+\'\'.join(f\'<li>{escape(s["dataset"])}: {escape(s["status"])} {escape(s["error"])}</li>\' for s in states)+\'</ul></html>\'\n    (output/\'report_all.html\').write_text(document, encoding=\'utf-8\')\n    return final_rows\n', 'phase4_data.py': '"""Paired-data, split integrity and metrics for Phase 4."""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport random\nimport re\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom skimage.metrics import structural_similarity\n\nEXT = {\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\', \'.tif\', \'.tiff\'}\n\n\ndef dump_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_csv(path, rows):\n    if not rows:\n        return\n    with Path(path).open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\nDATASET_SPECS = {\'o-hazy\': {\'label\': \'O-HAZY\',\n            \'relative_root\': \'dataset/O-HAZY\',\n            \'kind\': \'paired\',\n            \'can_train\': True,\n            \'val_ids\': [5, 15, 25, 35]},\n \'i-haze\': {\'label\': \'I-HAZE\',\n            \'relative_root\': \'dataset/I-HAZE\',\n            \'kind\': \'paired\',\n            \'can_train\': True,\n            \'val_ids\': []},\n \'reside6k\': {\'label\': \'RESIDE-6K\',\n              \'relative_root\': \'dataset/RESIDE-6K\',\n              \'kind\': \'paired\',\n              \'can_train\': True,\n              \'val_ids\': []},\n \'sots-its\': {\'label\': \'SOTS Indoor / ITS\',\n              \'relative_root\': \'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/indoor\',\n              \'kind\': \'evaluation-only\',\n              \'can_train\': False,\n              \'val_ids\': []},\n \'sots-ots\': {\'label\': \'SOTS Outdoor / OTS\',\n              \'relative_root\': \'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/outdoor\',\n              \'kind\': \'evaluation-only\',\n              \'can_train\': False,\n              \'val_ids\': []}}\n\n\ndef normalize_dataset_name(name):\n    key = str(name or \'o-hazy\').strip().lower().replace(\'_\', \'-\').replace(\'/\', \'-\').replace(\' \', \'-\')\n    key = re.sub(r\'-+\', \'-\', key).strip(\'-\')\n    aliases = {\'o-haze\': \'o-hazy\', \'ohazy\': \'o-hazy\', \'ihaze\': \'i-haze\', \'sots-indoor\': \'sots-its\', \'sots-indoor-its\': \'sots-its\',\n               \'its\': \'sots-its\', \'sots-outdoor\': \'sots-ots\', \'sots-outdoor-ots\': \'sots-ots\', \'ots\': \'sots-ots\', \'reside-6k\': \'reside6k\'}\n    key = aliases.get(key, key)\n    if key not in DATASET_SPECS:\n        raise ValueError(f\'DATASET_NAME không hợp lệ: {name}. Chọn một trong {list(DATASET_SPECS)}\')\n    return key\n\n\ndef _kind(path, root):\n    parents = {a.lower() for a in Path(path).relative_to(root).parts[:-1]}\n    stem = Path(path).stem.lower()\n    if parents & {\'gt\', \'clear\', \'clean\', \'groundtruth\', \'ground_truth\'} or stem.endswith((\'_gt\', \'_clear\', \'_clean\')):\n        return \'clear\'\n    if parents & {\'hazy\', \'haze\', \'input\', \'inputs\', \'hazy_images\'} or stem.endswith((\'_hazy\', \'_haze\', \'_input\')):\n        return \'hazy\'\n    return None\n\n\ndef _base_stem(path):\n    stem = Path(path).stem.lower()\n    stem = re.sub(r\'(?i)([_-]?hazy|[_-]?haze|[_-]?gt|[_-]?clear|[_-]?clean)$\', \'\', stem)\n    return re.sub(r\'\\d+\', lambda m: str(int(m[0])), stem)\n\n\ndef collect_pairs(root, dataset_name=\'o-hazy\'):\n    """Pair clear/hazy images across explicit split folders and common suffix conventions."""\n    dataset_name = normalize_dataset_name(dataset_name)\n    root = Path(root).resolve()\n    groups = {}\n    for p in sorted(root.rglob(\'*\')):\n        if not p.is_file() or p.suffix.lower() not in EXT:\n            continue\n        kind = _kind(p, root)\n        if kind is None:\n            continue\n        rel_parts = p.relative_to(root).parts\n        split = next((x.lower() for x in rel_parts if x.lower() in {\'train\', \'val\', \'validation\', \'test\'}), None)\n        if split == \'validation\': split = \'val\'\n        groups.setdefault(split or \'all\', {\'clear\': [], \'hazy\': []})[kind].append(p.resolve())\n    pairs = []\n    for split, sides in sorted(groups.items()):\n        exact, normalized = {}, {}\n        for clear in sides[\'clear\']:\n            for table, key in ((exact, clear.stem.lower()), (normalized, _base_stem(clear))):\n                if key in table:\n                    raise ValueError(f\'Trùng ảnh clear trong split {split}: {table[key]} và {clear}\')\n                table[key] = clear\n        used, ids = set(), set()\n        for hazy in sides[\'hazy\']:\n            clear = exact.get(hazy.stem.lower()) or normalized.get(_base_stem(hazy))\n            # RESIDE: one clear scene can have several atmospheric variants.\n            if clear is None and dataset_name in {\'reside6k\', \'sots-its\', \'sots-ots\'}:\n                clear = normalized.get(_base_stem(Path(hazy.stem.split(\'_\')[0])))\n            if clear is None:\n                raise ValueError(f\'Không ghép được ảnh hazy với clear trong split {split}: {hazy}\')\n            sid = hazy.stem.lower()\n            if sid in ids:\n                raise ValueError(f\'Trùng ID hazy trong split {split}: {sid}\')\n            ids.add(sid); used.add(clear)\n            pairs.append({\'id\': sid, \'scene\': _base_stem(clear), \'hazy\': str(hazy), \'clear\': str(clear),\n                          \'source_split\': None if split == \'all\' else split})\n        if set(sides[\'clear\']) != used:\n            raise ValueError(f\'Có ảnh clear thiếu hazy trong split {split}: {set(sides["clear"])-used}\')\n    if not pairs:\n        raise ValueError(f\'Không tìm thấy cặp clear/hazy trong {root}\')\n    if dataset_name == \'o-hazy\':\n        # Phase 3 protocol: IDs 1-40 for tuning, 41-45 held out for test.\n        def numeric(row):\n            m = re.match(r\'0*(\\d+)\', row[\'id\'])\n            return int(m.group(1)) if m else None\n        by_id = {}\n        for row in pairs:\n            sid = numeric(row)\n            if sid in by_id or sid is None:\n                raise ValueError(f\'Trùng hoặc thiếu ID scene O-HAZY: {row}\')\n            by_id[sid] = row\n        if set(by_id) != set(range(1, 46)):\n            raise ValueError(\'O-HAZY cần đủ 45 scene 01–45 để giữ protocol Phase 3.\')\n        pairs = [by_id[i] | {\'id\': i} for i in sorted(by_id)]\n    return pairs\n\n\ndef _content_groups(rows):\n    """Keep shared scene IDs or identical clear/hazy files in one component."""\n    parents, owners, hashes = {}, {}, {}\n\n    def find(scene):\n        while parents[scene] != scene:\n            parents[scene] = parents[parents[scene]]\n            scene = parents[scene]\n        return scene\n\n    for row in rows:\n        scene = row[\'scene\']; parents.setdefault(scene, scene)\n        for side in (\'clear\', \'hazy\'):\n            path = row[side]\n            if path not in hashes: hashes[path] = sha256(path)\n            key = (side, hashes[path])\n            other = owners.setdefault(key, scene)\n            a, b = find(scene), find(other)\n            if a != b:\n                # Stable representative regardless of file traversal order.\n                parents[max(a, b)] = min(a, b)\n    return [find(row[\'scene\']) for row in rows]\n\n\ndef _holdout_train(rows, seed, fraction, max_scenes=0, group_content=False):\n    groups = _content_groups(rows) if group_content else [r[\'scene\'] for r in rows]\n    scenes = sorted(set(groups))\n    if len(scenes) < 2:\n        raise ValueError(\'Cần ít nhất hai scene train để tách validation.\')\n    random.Random(seed).shuffle(scenes)\n    count = min(len(scenes)-1, max(1, round(len(scenes)*fraction)))\n    if max_scenes: count = min(count, max_scenes)\n    val = set(scenes[:count])\n    return ([r for r, group in zip(rows, groups) if group not in val],\n            [r for r, group in zip(rows, groups) if group in val])\n\n\ndef partition(pairs, cfg):\n    dataset_name = normalize_dataset_name(cfg.get(\'dataset_name\', \'o-hazy\'))\n    if dataset_name == \'o-hazy\' and all(isinstance(p[\'id\'], int) for p in pairs):\n        val = set(cfg.get(\'val_ids\', [5, 15, 25, 35]))\n        if not val or not val <= set(range(1, 41)):\n            raise ValueError(\'val_ids O-HAZY phải thuộc 1–40 và không rỗng.\')\n        result = {\'train\': [p for p in pairs if p[\'id\'] <= 40 and p[\'id\'] not in val],\n                  \'val\': [p for p in pairs if p[\'id\'] in val],\n                  \'test\': [p for p in pairs if 41 <= p[\'id\'] <= 45]}\n    else:\n        explicit = {p.get(\'source_split\') for p in pairs}\n        if None in explicit and len(explicit) > 1:\n            raise ValueError(\'Không trộn dữ liệu phẳng với thư mục train/val/test.\')\n        if dataset_name.startswith(\'sots-\'):\n            # Some repackaged SOTS have train/val folders. They are still benchmark test images.\n            result = {\'train\': [], \'val\': [], \'test\': list(pairs)}\n        elif dataset_name == \'reside6k\' and cfg.get(\'eval_only\') and explicit == {None}:\n            # A downloaded test/clear + test/hazy root is flat relative to itself.\n            result = {\'train\': [], \'val\': [], \'test\': list(pairs)}\n        elif explicit == {None}:\n            raise ValueError(\'Dataset train cần thư mục train và test; không tự lấy test làm validation.\')\n        else:\n            result = {k: [p for p in pairs if p.get(\'source_split\') == k] for k in (\'train\', \'val\', \'test\')}\n            if result[\'train\'] and not result[\'val\']:\n                result[\'train\'], result[\'val\'] = _holdout_train(\n                    result[\'train\'], cfg.get(\'seed\', 2026), .1, group_content=dataset_name == \'reside6k\')\n    assert_disjoint(result)\n    return result\n\n\ndef assert_disjoint(splits):\n    seen = {}\n    for split, rows in splits.items():\n        ids = set()\n        for row in rows:\n            if row[\'id\'] in ids:\n                raise ValueError(f\'Trùng ID trong split {split}: {row["id"]}\')\n            ids.add(row[\'id\'])\n            keys = [(\'scene\', row[\'scene\']), (\'path\', row[\'clear\']), (\'path\', row[\'hazy\'])]\n            keys += [(\'clear_sha256\', row[\'clear_sha256\'])] if row.get(\'clear_sha256\') else []\n            keys += [(\'hazy_sha256\', row[\'hazy_sha256\'])] if row.get(\'hazy_sha256\') else []\n            for key in keys:\n                if key in seen and seen[key] != split:\n                    raise ValueError(f\'Rò rỉ dữ liệu giữa {seen[key]} và {split}: {key}\')\n                seen[key] = split\n\n\ndef dataset_splits(cfg):\n    name = normalize_dataset_name(cfg.get(\'dataset_name\', \'o-hazy\'))\n    data = Path(cfg.get(\'data_root\') or Path(cfg[\'project_root\']) / DATASET_SPECS[name][\'relative_root\']).expanduser().resolve()\n    if not data.is_dir():\n        raise FileNotFoundError(f\'Không có dataset {name} tại {data}. Đặt DATA_ROOT_OVERRIDE tới dữ liệu có sẵn.\')\n    splits = partition(collect_pairs(data, name), {**cfg, \'dataset_name\': name})\n    if cfg.get(\'train_root\'):\n        if not name.startswith(\'sots-\'):\n            raise ValueError(\'TRAIN_ROOT chỉ dành cho train ITS/OTS riêng khi đánh giá trên SOTS.\')\n        train_root = Path(cfg[\'train_root\']).expanduser().resolve()\n        if train_root == data or train_root.is_relative_to(data) or data.is_relative_to(train_root):\n            raise ValueError(\'TRAIN_ROOT phải tách biệt với thư mục benchmark SOTS.\')\n        training = collect_pairs(train_root, name)\n        if cfg.get(\'exclude_benchmark_overlap\'):\n            # Integrity filtering only; never use test predictions/quality to select training data.\n            scenes = {r[\'scene\'] for r in splits[\'test\']}\n            test_hashes = {sha256(r[k]) for r in splits[\'test\'] for k in (\'clear\', \'hazy\')}\n            hashes = {}\n            blocked = set(scenes)\n            for row in training:\n                for key in (\'clear\', \'hazy\'):\n                    if row[key] not in hashes: hashes[row[key]] = sha256(row[key])\n                    if hashes[row[key]] in test_hashes: blocked.add(row[\'scene\'])\n            before = len(training)\n            training = [r for r in training if r[\'scene\'] not in blocked]\n            print(f\'Loại {before-len(training)} cặp train trùng scene/nội dung với benchmark.\', flush=True)\n        explicit = {r.get(\'source_split\') for r in training}\n        if explicit == {None}:\n            splits[\'train\'], splits[\'val\'] = _holdout_train(\n                training, cfg.get(\'seed\', 2026), .1, cfg.get(\'val_max_scenes\', 0), group_content=True)\n        elif explicit <= {\'train\', \'val\'} and \'train\' in explicit:\n            splits[\'train\'] = [r for r in training if r[\'source_split\'] == \'train\']\n            splits[\'val\'] = [r for r in training if r[\'source_split\'] == \'val\']\n            if not splits[\'val\']:\n                splits[\'train\'], splits[\'val\'] = _holdout_train(\n                    splits[\'train\'], cfg.get(\'seed\', 2026), .1, cfg.get(\'val_max_scenes\', 0), group_content=True)\n        else:\n            raise ValueError(\'TRAIN_ROOT chỉ chứa train/val hoặc các cặp ảnh phẳng, không chứa test.\')\n    # Discard excess validation scenes/variants, never move them into training.\n    cap = cfg.get(\'val_max_scenes\', 0)\n    if cap:\n        scenes = sorted({r[\'scene\'] for r in splits[\'val\']})\n        random.Random(cfg.get(\'seed\', 2026)).shuffle(scenes)\n        chosen = set(scenes[:cap])\n        splits[\'val\'] = [r for r in splits[\'val\'] if r[\'scene\'] in chosen]\n    pair_cap = cfg.get(\'val_max_pairs\', 0)\n    if pair_cap and len(splits[\'val\']) > pair_cap:\n        rows = list(splits[\'val\']); random.Random(cfg.get(\'seed\', 2026)).shuffle(rows)\n        splits[\'val\'] = rows[:pair_cap]\n    requested = cfg.get(\'eval_only\')\n    eval_only = (not splits[\'train\']) if requested is None else bool(requested)\n    if not splits[\'test\'] or (not eval_only and (not splits[\'train\'] or not splits[\'val\'])):\n        raise ValueError(\'Cần test để đánh giá; fine-tune còn cần train và validation riêng. SOTS cần TRAIN_ROOT để train.\')\n    assert_disjoint(splits)\n    return data, splits, eval_only\n\n\ndef manifest_digest(splits):\n    rows = [{k: row[k] for k in (\'split\', \'id\', \'scene\', \'hazy_sha256\', \'clear_sha256\')}\n            for split in (\'train\', \'val\', \'test\') for row in splits[split]]\n    return hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()\n\n\ndef prepare(cfg):\n    work = Path(cfg[\'work_root\'])\n    work.mkdir(parents=True, exist_ok=True)\n    root, splits, evaluation_only = dataset_splits(cfg)\n    name = normalize_dataset_name(cfg.get(\'dataset_name\', \'o-hazy\'))\n    # Hash shared clear images once, including synthetic sets with many atmospheric variants.\n    hashes = {}\n    for split, rows in splits.items():\n        for row in rows:\n            row.update(dataset_name=name, split=split)\n            for key in (\'hazy\', \'clear\'):\n                if row[key] not in hashes:\n                    hashes[row[key]] = sha256(row[key])\n                row[key+\'_sha256\'] = hashes[row[key]]\n    assert_disjoint(splits)\n    value = {\'dataset_name\': name, \'data_root\': str(root), \'evaluation_only\': evaluation_only,\n             \'splits\': splits, \'manifest_sha256\': manifest_digest(splits)}\n    dump_json(work/\'inputs.json\', value)\n    write_csv(work/\'split_manifest.csv\', [r for rows in splits.values() for r in rows])\n    print(\'Dataset:\', name, {k: len(v) for k, v in splits.items()}, \'eval_only:\', evaluation_only, flush=True)\n    return value\n\n\ndef validate(cfg, inputs):\n    root, splits, only = dataset_splits(cfg)\n    if inputs[\'dataset_name\'] != normalize_dataset_name(cfg[\'dataset_name\']) or str(root) != inputs[\'data_root\']:\n        raise ValueError(\'Dataset/root đã đổi; chạy prepare lại.\')\n    if only != inputs[\'evaluation_only\']:\n        raise ValueError(\'Chế độ dataset đã đổi; chạy prepare lại.\')\n    for split in splits:\n        keys = lambda rows: [(r[\'id\'], r[\'hazy\'], r[\'clear\']) for r in rows]\n        if keys(splits[split]) != keys(inputs[\'splits\'][split]):\n            raise ValueError(\'Split đã đổi; chạy prepare lại.\')\n    checked = {}\n    for rows in inputs[\'splits\'].values():\n        for row in rows:\n            for key in (\'hazy\', \'clear\'):\n                if row[key] not in checked:\n                    checked[row[key]] = sha256(row[key])\n                if checked[row[key]] != row[key+\'_sha256\']:\n                    raise ValueError(f\'Dữ liệu đã đổi: {row[key]}\')\n    assert_disjoint(inputs[\'splits\'])\n    if manifest_digest(inputs[\'splits\']) != inputs[\'manifest_sha256\']:\n        raise ValueError(\'Manifest không còn khớp checksum.\')\n\n\ndef load_pair(pair, max_side):\n    images = []\n    for key in (\'hazy\', \'clear\'):\n        image = cv2.imread(pair[key], cv2.IMREAD_COLOR)\n        if image is None:\n            raise OSError(f\'Không đọc được {pair[key]}\')\n        images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))\n    x, y = images\n    if (pair.get(\'dataset_name\') == \'sots-its\' and y.shape[:2] == (x.shape[0]+20, x.shape[1]+20)):\n        # Standard SOTS indoor GT includes a 10-pixel frame absent from the hazy image.\n        y = y[10:-10, 10:-10]\n    if x.shape != y.shape:\n        raise ValueError(f\'Cặp ảnh khác shape: scene {pair["id"]}; {x.shape}, {y.shape}\')\n    h, w = x.shape[:2]\n    if max_side and max(h, w) > max_side:\n        scale = max_side / max(h, w)\n        size = (round(w * scale), round(h * scale))\n        x, y = [cv2.resize(a, size, interpolation=cv2.INTER_AREA) for a in (x, y)]\n    return [torch.from_numpy(a.copy()).permute(2, 0, 1).float().div_(255) for a in (x, y)]\n\n\ndef pad32(x):\n    h, w = x.shape[-2:]\n    dh, dw = (-h) % 32, (-w) % 32\n    return F.pad(x, (0, dw, 0, dh), mode=\'reflect\' if dh < h and dw < w else \'replicate\')\n\n\ndef final_output(y):\n    return y[-1] if isinstance(y, (tuple, list)) else y\n\n\ndef metrics(pred, gt, border=4):\n    a, b = [t.detach().float().clamp(0, 1).cpu().numpy().transpose(1, 2, 0) for t in (pred, gt)]\n    if border:\n        a, b = [v[border:-border, border:-border] for v in (a, b)]\n    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)\n    psnr = float(-10 * np.log10(max(mse, 1e-12)))\n    # Same 11x11 Gaussian window/sigma and valid-region averaging as BasicSR SSIM.\n    ssim = float(structural_similarity(a, b, channel_axis=2, data_range=1.,\n                                      gaussian_weights=True, sigma=1.5, use_sample_covariance=False))\n    return psnr, ssim\n\n\n\ndef metric_values(pred, gt, border=4):\n    if border < 0 or min(pred.shape[-2:]) <= 2*border+10:\n        raise ValueError(\'Vùng đánh giá quá nhỏ cho cửa sổ SSIM 11x11.\')\n    if not torch.isfinite(pred).all() or not torch.isfinite(gt).all():\n        raise FloatingPointError(\'Ảnh đánh giá chứa NaN/Inf.\')\n    psnr, ssim = metrics(pred, gt, border)\n    a, b = [v.detach().double().clamp(0, 1).cpu() for v in (pred, gt)]\n    if border:\n        a, b = [v[..., border:-border, border:-border] for v in (a, b)]\n    mse = float((a-b).square().mean())\n    return {\'PSNR\': psnr, \'SSIM\': ssim, \'MSE\': mse, \'values\': a.numel()}\n\n\ndef efficiency(fps, mse):\n    # Infinity for a perfect reconstruction; no arbitrary epsilon alters rankings.\n    if not np.isfinite(fps) or fps <= 0 or not np.isfinite(mse) or mse < 0:\n        raise ValueError(\'FPS phải hữu hạn và dương; MSE phải hữu hạn và không âm.\')\n    return float(\'inf\') if mse == 0 else float(fps/mse)\n', 'phase4_download.py': '"""Resolve existing datasets or download the external synthetic benchmarks."""\nfrom pathlib import Path\nimport json\n\n\ndef pair_root(root):\n    root = Path(root)\n    dirs = {p.name.lower(): p for p in root.iterdir() if p.is_dir()} if root.is_dir() else {}\n    clear = next((dirs[k] for k in (\'clear\', \'gt\', \'clean\', \'groundtruth\', \'ground_truth\') if k in dirs), None)\n    hazy = next((dirs[k] for k in (\'hazy\', \'haze\', \'input\', \'inputs\', \'hazy_images\') if k in dirs), None)\n    return clear is not None and hazy is not None and any(p.is_file() for p in clear.iterdir()) and any(p.is_file() for p in hazy.iterdir())\n\n\ndef find_dataset(base, name, test_only=False):\n    base = Path(base).expanduser().resolve()\n    candidates = [base, *sorted((p for p in base.rglob(\'*\') if p.is_dir()), key=lambda p: (len(p.parts), str(p)))]\n    if name.startswith(\'sots-\'):\n        domain = \'indoor\' if name == \'sots-its\' else \'outdoor\'\n        candidates = [p for p in candidates if domain in {x.lower() for x in p.parts}]\n    if name == \'reside6k\':\n        if not test_only:\n            for p in candidates:\n                children = {d.name.lower() for d in p.iterdir() if d.is_dir()}\n                if {\'train\', \'test\'} <= children:\n                    return p\n        tests = [p for p in candidates if \'test\' in {x.lower() for x in p.parts} and pair_root(p)]\n        if tests:\n            return tests[0]\n        # Never silently evaluate a training directory when the test layout is missing.\n        raise FileNotFoundError(f\'Không tìm thấy RESIDE-6K test/clear và test/hazy dưới {base}.\')\n    for p in candidates:\n        if pair_root(p):\n            return p\n    raise FileNotFoundError(f\'Không tìm thấy cặp clear/hazy cho {name} dưới {base}.\')\n\n\ndef resolve_datasets(names, specs, project_root, overrides, cache_root, *,\n                     allow_download=True, test_only=False, download=None,\n                     reside_handle=\'kmljts/reside-6k\',\n                     sots_handle=\'balraj98/synthetic-objective-testing-set-sots-reside\',\n                     continue_on_error=False):\n    """Return actual roots plus errors; an explicit local override never triggers a download."""\n    project_root, cache_root = Path(project_root), Path(cache_root)\n    cache_root.mkdir(parents=True, exist_ok=True)\n    manifest = cache_root/\'downloads.json\'\n    cached = json.loads(manifest.read_text(encoding=\'utf-8\')) if manifest.is_file() else {}\n    roots, errors, downloads = {}, {}, {}\n    for name in names:\n        try:\n            explicit = overrides.get(name)\n            local = Path(explicit).expanduser().resolve() if explicit else project_root/specs[name][\'relative_root\']\n            if explicit:\n                if not local.is_dir():\n                    raise FileNotFoundError(f\'Đường dẫn local của {name} không tồn tại: {local}\')\n                roots[name] = str(local)\n                continue\n            if name in {\'o-hazy\', \'i-haze\'}:\n                if not local.is_dir():\n                    raise FileNotFoundError(f\'{name} phải lấy từ checkout VDT_GD2: {local}\')\n                roots[name] = str(local.resolve())\n                continue\n            handle = reside_handle if name == \'reside6k\' else sots_handle\n            if local.is_dir():\n                try:\n                    roots[name] = str(find_dataset(local, name, test_only))\n                    continue\n                except FileNotFoundError:\n                    pass\n            if handle in cached and Path(cached[handle]).is_dir():\n                try:\n                    roots[name] = str(find_dataset(cached[handle], name, test_only))\n                    continue\n                except FileNotFoundError:\n                    pass\n            if not allow_download:\n                raise FileNotFoundError(f\'Thiếu {name}; bật AUTO_DOWNLOAD_DATASETS hoặc đặt DATA_ROOT_OVERRIDE(S).\')\n            if handle not in downloads:\n                print(f\'Đang tải/tái dùng cache KaggleHub: {handle}\', flush=True)\n                if download is None:\n                    import kagglehub\n                    downloads[handle] = Path(kagglehub.dataset_download(handle))\n                else:\n                    downloads[handle] = Path(download(handle))\n                cached[handle] = str(downloads[handle].resolve())\n                manifest.write_text(json.dumps(cached, indent=2), encoding=\'utf-8\')\n            roots[name] = str(find_dataset(downloads[handle], name, test_only))\n        except Exception as exc:\n            if not continue_on_error:\n                raise\n            errors[name] = str(exc)\n            print(f\'{name}: chuẩn bị dataset lỗi: {exc}\', flush=True)\n    return roots, errors\n\n\ndef resolve_training_roots(names, specs, project_root, train_roots, overrides, cache_root,\n                           *, enabled=True, allow_download=True, download=None,\n                           reside_handle=\'kmljts/reside-6k\'):\n    """Resolve separate training data; never substitute a SOTS benchmark directory."""\n    roots = dict(train_roots)\n    missing = [n for n in names if n.startswith(\'sots-\') and not roots.get(n)]\n    if not enabled or not missing:\n        return roots\n    resolved, _ = resolve_datasets([\'reside6k\'], specs, project_root, overrides, cache_root,\n        allow_download=allow_download, test_only=False, download=download,\n        reside_handle=reside_handle)\n    base = Path(resolved[\'reside6k\'])\n    train = next((p for p in base.iterdir() if p.is_dir() and p.name.lower() == \'train\'), None)\n    if train is None or not pair_root(train):\n        raise FileNotFoundError(f\'Cần RESIDE-6K/train với các cặp GT/hazy, không dùng test: {base}\')\n    for name in missing:\n        roots[name] = str(train.resolve())\n    return roots\n', 'phase4_profile.py': '"""Analytical operation estimates; shared counting rules for every model."""\nfrom collections import defaultdict\nimport math\nimport torch\nfrom torch import nn\n\ndef profile_gflops(model, x):\n    """Analytical forward hooks; 1 multiply-add = 2 FLOPs, not GFLOP/s.\n\n    Count Conv/Linear arithmetic exactly under dense zero-padding conventions.\n    Functional SS2D projections are counted explicitly. Scan, FFT and Haar are\n    supplemental estimates; activations, normalization, pooling and elementwise\n    bookkeeping are not counted. This is not a complete hardware instruction count.\n    """\n    rows = defaultdict(float)\n    handles = []\n\n    def hook(module, args, result):\n        a = args[0]\n        name = module.__class__.__name__\n        if isinstance(module, (nn.Conv1d, nn.Conv2d)):\n            rows[\'counted_conv_linear\'] += 2*result.numel()*(module.in_channels//module.groups)*math.prod(module.kernel_size)\n        elif isinstance(module, nn.ConvTranspose2d):\n            rows[\'counted_conv_linear\'] += 2*a.numel()*(module.out_channels//module.groups)*math.prod(module.kernel_size)\n        elif isinstance(module, nn.Linear):\n            rows[\'counted_conv_linear\'] += 2*a.numel()*module.out_features\n        elif name == \'SS2D\':\n            b, h, w, _ = a.shape\n            d, n, r = module.d_inner, module.d_state, module.dt_rank\n            rows[\'counted_ss2d_einsum\'] += 2*b*4*h*w*d*(2*r+2*n)\n            rows[\'estimated_selective_scan\'] += b*4*h*w*d*(8*n+2)\n        elif name == \'fft_bench\':\n            b, c, h, w = a.shape\n            rows[\'estimated_fft\'] += 10*b*c*h*w*math.log2(h*w)\n        elif name == \'DWTForward\':\n            # pytorch_wavelets functional Haar convolutions bypass nn.Conv hooks.\n            rows[\'estimated_internal_haar\'] += 4*a.numel()\n\n    was_training = {m: m.training for m in model.modules()}\n    try:\n        model.eval()\n        for module in model.modules():\n            handles.append(module.register_forward_hook(hook))\n        with torch.inference_mode():\n            model(x)\n    finally:\n        for handle in handles:\n            handle.remove()\n        for module, training in was_training.items():\n            module.training = training\n    # Outer DWT/IWT are constructed inside forward and cannot be hooked persistently.\n    if any(hasattr(m, \'restoration_network\') for m in model.modules()):\n        h, w = x.shape[-2:]\n        padded_values = math.prod(x.shape[:-2]) * ((h+31)//32*32) * ((w+31)//32*32)\n        rows[\'estimated_outer_haar\'] += 8*padded_values\n    counted = sum(v for k, v in rows.items() if k.startswith(\'counted_\'))/1e9\n    supplemental = sum(v for k, v in rows.items() if k.startswith(\'estimated_\'))/1e9\n    return {\'counted_GFLOPs\': counted, \'supplemental_GFLOPs\': supplemental,\n            \'estimated_total_GFLOPs\': counted+supplemental,\n            **{k+\'_GFLOPs\': v/1e9 for k, v in rows.items()}}\n', 'phase4_runner.py': '"""Phase 4: reproducible adaptation/training and same-session quality/speed evaluation."""\nfrom pathlib import Path\nfrom collections import OrderedDict\nimport copy\nimport hashlib\nimport html\nimport json\nimport math\nimport random\nimport re\nimport sys\nimport time\nimport zipfile\nimport cv2\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nimport phase4_data as data\nimport phase4_arch as arch\nfrom phase4_profile import profile_gflops\n\n\ndef atomic_save(value, path):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix+\'.tmp\')\n    torch.save(value, tmp); tmp.replace(path)\n\n\ndef configure(cfg):\n    random.seed(cfg[\'seed\']); np.random.seed(cfg[\'seed\']); torch.manual_seed(cfg[\'seed\'])\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg[\'seed\'])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    cv2.setNumThreads(0)\n\n\ndef device_for(cfg):\n    device = torch.device(cfg.get(\'device\', \'cuda\'))\n    if device.type == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Cần GPU CUDA; CPU chỉ dùng cho kiểm thử.\')\n    return device\n\n\ndef source_hashes(cfg):\n    here = Path(__file__).parent\n    paths = list(here.glob(\'phase4_*.py\'))\n    paths += list((Path(cfg[\'repo_root\'])/\'basicsr/archs\').glob(\'*.py\'))\n    return {p.name: data.sha256(p) for p in sorted(paths)}\n\n\ndef resolve_source(cfg):\n    name = data.normalize_dataset_name(cfg[\'dataset_name\'])\n    if name == \'i-haze\':\n        if cfg.get(\'checkpoint\') or cfg.get(\'student_checkpoint\'):\n            raise ValueError(\'I-HAZE phase 4 yêu cầu khởi tạo ngẫu nhiên; để trống cả hai checkpoint.\')\n        return None\n    if cfg.get(\'checkpoint\'):\n        path = Path(cfg[\'checkpoint\']).expanduser().resolve()\n        if not path.is_file(): raise FileNotFoundError(path)\n        return path\n    if name in (\'sots-its\', \'sots-ots\', \'reside6k\') and cfg.get(\'reside_checkpoint\'):\n        path = Path(cfg[\'reside_checkpoint\']).expanduser().resolve()\n        if not path.is_file(): raise FileNotFoundError(path)\n        return path\n    filenames = [\'Ohaze_1200_1600_27.22.pth\'] if name == \'o-hazy\' else [\'reside6k_32.15.pth\']\n    roots = [Path(cfg[\'project_root\']), Path(cfg[\'work_root\'])/\'weights\']\n    roots += [Path(p).expanduser() for p in cfg.get(\'checkpoint_search_roots\', []) if p]\n    for root in roots:\n        if root.is_dir():\n            for filename in filenames:\n                found = sorted(root.rglob(filename))\n                if found: return found[0].resolve()\n    archive = Path(cfg[\'checkpoint_zip\']).expanduser() if cfg.get(\'checkpoint_zip\') else None\n    if archive is None and cfg.get(\'allow_weight_download\', False):\n        archive = Path(cfg[\'work_root\'])/\'WDMamba_ckpts.zip\'\n        if not archive.is_file():\n            import gdown\n            tmp = archive.with_suffix(\'.zip.partial\')\n            result = gdown.download(id=\'1Vm-1mtMSc9YIUICq4nWKu-4AHCtYnPpF\', output=str(tmp),\n                                    quiet=False, use_cookies=False, resume=True)\n            if result is None or not zipfile.is_zipfile(tmp):\n                raise RuntimeError(\'Không tải được ZIP trọng số; đặt CHECKPOINT hoặc CHECKPOINT_ZIP.\')\n            tmp.replace(archive)\n    if archive:\n        target = (Path(cfg[\'work_root\'])/\'weights\').resolve(); target.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(archive) as z:\n            if z.testzip() is not None: raise ValueError(\'ZIP trọng số lỗi.\')\n            for item in z.infolist():\n                if not (target/item.filename).resolve().is_relative_to(target):\n                    raise ValueError(\'ZIP chứa đường dẫn nằm ngoài thư mục giải nén.\')\n            z.extractall(target)\n        for filename in filenames:\n            found = sorted(target.rglob(filename))\n            if found: return found[0].resolve()\n    # O-HAZY can still improve a local Phase 3 model. Its scope is logged explicitly;\n    # gains must then be measured against that model, not against the absent teacher.\n    if name == \'o-hazy\':\n        for phase in (\'phase 3.1\', \'phase 3\'):\n            p = Path(cfg[\'project_root\'])/\'solution/Haze Clear\'/phase/\'fast_wdmamba_o_hazy_best.pth\'\n            if p.is_file(): return p.resolve()\n    raise FileNotFoundError(f\'Không tìm thấy trọng số cho {name}. Đặt CHECKPOINT riêng cho Indoor/Outdoor \'\n                            \'hoặc CHECKPOINT_ZIP. Không tự dùng trọng số O-HAZY cho SOTS.\')\n\n\ndef prepare(cfg):\n    inputs = data.prepare(cfg)\n    require_adaptation(cfg, inputs)\n    source = resolve_source(cfg)\n    student = Path(cfg[\'student_checkpoint\']).expanduser().resolve() if cfg.get(\'student_checkpoint\') else None\n    if student is None and inputs[\'dataset_name\'] == \'o-hazy\' and source is not None:\n        # Reuse the recovered Phase 3 DEN when its recorded teacher matches the\n        # full reference. Avoid starting again from destructive pruning.\n        digest = data.sha256(source)\n        for phase in (\'phase 3.1\', \'phase 3\'):\n            candidate = Path(cfg[\'project_root\'])/\'solution/Haze Clear\'/phase/\'fast_wdmamba_o_hazy_best.pth\'\n            if candidate.is_file() and candidate.resolve() != source.resolve():\n                obj = torch.load(candidate, map_location=\'cpu\', weights_only=False)\n                if obj.get(\'teacher_sha256\') == digest:\n                    student = candidate.resolve(); break\n    if student is not None and not student.is_file(): raise FileNotFoundError(student)\n    if student is not None:\n        obj = torch.load(student, map_location=\'cpu\', weights_only=False)\n        previous = obj.get(\'config\', {})\n        previous_name = data.normalize_dataset_name(previous.get(\'dataset_name\', \'o-hazy\'))\n        if previous_name != inputs[\'dataset_name\']:\n            raise ValueError(f\'Student {previous_name} không cùng dataset với {inputs["dataset_name"]}.\')\n        # No cross-teacher O-HAZY mix when the student provides a provenance hash.\n        if source and arch.detect_den_depth(arch.checkpoint_state(source)) > 1:\n            if obj.get(\'teacher_sha256\') and obj[\'teacher_sha256\'] != data.sha256(source):\n                raise ValueError(\'Teacher SHA256 của student không khớp model tham chiếu.\')\n    inputs.update(source=str(source) if source else \'\', source_sha256=data.sha256(source) if source else \'\',\n                  student_source=str(student) if student else \'\', student_sha256=data.sha256(student) if student else \'\',\n                  code_sha256=source_hashes(cfg))\n    path = Path(cfg[\'work_root\'])/\'inputs.json\'\n    # Preserve identity of a previous run; changing inputs must use a new RUN_NAME.\n    output = Path(cfg[\'output_root\'])\n    for artifact in (output/\'final_best.pth\', output/\'reference_best.pth\'):\n        if artifact.is_file():\n            saved = torch.load(artifact, map_location=\'cpu\', weights_only=False)\n            if saved[\'identity\'] != identity(cfg, inputs):\n                raise ValueError(\'Run đã có trọng số nhưng dữ liệu/cấu hình/mã khác. Đổi RUN_NAME.\')\n    data.dump_json(path, inputs)\n    return inputs\n\n\ndef identity(cfg, inputs):\n    keys = (\'dataset_name\', \'seed\', \'den_blocks\', \'detail_width\', \'reference_depth\', \'train_max_side\',\n            \'eval_max_side\', \'crop_border\', \'adapt_schedule\', \'scratch_schedule\', \'reference_schedule\',\n            \'l1_weight\', \'edge_weight\', \'kd_weight\', \'ema_decay\', \'weight_decay\', \'val_every\', \'patience\')\n    return {\'config\': {**{k: cfg[k] for k in keys},\n                       \'runtime_only_datasets\': cfg.get(\'runtime_only_datasets\', []),\n                       \'require_compression\': cfg.get(\'require_compression\', False)},\n            \'manifest\': inputs[\'manifest_sha256\'],\n            \'source\': inputs[\'source_sha256\'], \'student\': inputs[\'student_sha256\'], \'code\': inputs[\'code_sha256\']}\n\n\ndef require_adaptation(cfg, inputs):\n    if cfg.get(\'require_compression\') and (\n            inputs[\'evaluation_only\'] or cfg[\'dataset_name\'] in cfg.get(\'runtime_only_datasets\', [])):\n        raise ValueError(\'REQUIRE_COMPRESSION=True cần train/val riêng để tạo model nhẹ. \'\n                         \'SOTS cần TRAIN_ROOT hoặc AUTO_RESIDE_TRAIN=True; không dùng bản sao Reference.\')\n\n\ndef compression_info(reference, final):\n    ref_count = sum(p.numel() for p in reference.parameters())\n    final_count = sum(p.numel() for p in final.parameters())\n    return {\'reference_params\': ref_count, \'final_params\': final_count,\n            \'parameter_reduction\': 1-final_count/max(1, ref_count),\n            \'architecture_changed\': reference.spec != final.spec,\n            \'compressed\': reference.spec != final.spec and final_count < ref_count}\n\n\ndef adaptation_start(cfg, inputs, reference):\n    start = arch.reference_model(cfg[\'repo_root\'], inputs[\'student_source\']) if inputs[\'student_source\'] else reference\n    if start.spec[\'depth\'] < cfg[\'den_blocks\']:\n        print(\'Student không đủ block; chuyển trọng số từ teacher đầy đủ.\', flush=True)\n        start = reference\n    return start\n\n\ndef validate(cfg, inputs):\n    data.validate(cfg, inputs)\n    for path_key, hash_key in ((\'source\', \'source_sha256\'), (\'student_source\', \'student_sha256\')):\n        if inputs.get(path_key) and data.sha256(inputs[path_key]) != inputs[hash_key]:\n            raise ValueError(f\'File đã đổi: {inputs[path_key]}\')\n    if source_hashes(cfg) != inputs[\'code_sha256\']:\n        raise ValueError(\'Mã đã đổi sau prepare. Chạy prepare với RUN_NAME mới nếu đã train.\')\n\n\n@torch.no_grad()\ndef quality(model, rows, cfg, device, save_dir=None, progress_label=None):\n    model.eval(); result = []; started = time.perf_counter()\n    if progress_label: print(f\'{progress_label}: PSNR/SSIM/MSE 0/{len(rows)} ảnh\', flush=True)\n    for index, row in enumerate(rows, 1):\n        x, gt = data.load_pair(row, cfg[\'eval_max_side\'])\n        pred = model(x[None].to(device))[0].cpu()\n        values = data.metric_values(pred, gt, cfg[\'crop_border\'])\n        result.append({\'dataset\': cfg[\'dataset_name\'], \'id\': row[\'id\'], \'height\': x.shape[-2],\n                       \'width\': x.shape[-1], **values})\n        if save_dir:\n            folder = Path(save_dir); folder.mkdir(parents=True, exist_ok=True)\n            image = (pred.clamp(0, 1).permute(1, 2, 0).numpy()*255).round().astype(np.uint8)\n            name = re.sub(r\'[^\\w.-]\', \'_\', str(row[\'id\']))\n            cv2.imwrite(str(folder/f\'{name}.png\'), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))\n        if progress_label and (index == 1 or index % 25 == 0 or index == len(rows)):\n            print(f\'{progress_label}: PSNR/SSIM/MSE {index}/{len(rows)} ảnh; \'\n                  f\'{time.perf_counter()-started:.1f}s\', flush=True)\n    return result\n\n\ndef metric_mean(rows, key):\n    return float(np.mean([r[key] for r in rows]))\n\n\n@torch.no_grad()\ndef latency(model, x, cfg):\n    model.eval()\n    for _ in range(cfg[\'warmup\']): model(x)\n    values = []\n    if x.is_cuda:\n        torch.cuda.synchronize(x.device); torch.cuda.reset_peak_memory_stats(x.device)\n    for _ in range(cfg[\'timing_repeats\']):\n        if x.is_cuda:\n            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\n            start.record(); model(x); end.record(); end.synchronize()\n            values.append(start.elapsed_time(end))\n        else:\n            start = time.perf_counter(); model(x); values.append((time.perf_counter()-start)*1000)\n    if not values: raise ValueError(\'timing_repeats phải dương.\')\n    mean = float(np.mean(values))\n    return {\'latency_ms\': mean, \'FPS\': 1000/mean, \'p50_ms\': float(np.median(values)),\n            \'p95_ms\': float(np.percentile(values, 95)),\n            \'peak_allocated_GiB\': torch.cuda.max_memory_allocated(x.device)/2**30 if x.is_cuda else None}\n\n\nclass PairCache:\n    def __init__(self, rows, cfg):\n        self.rows, self.cfg, self.cache = rows, cfg, OrderedDict()\n\n    def sample(self, crop):\n        row = random.choice(self.rows); key = row[\'hazy\']\n        if key not in self.cache:\n            self.cache[key] = data.load_pair(row, self.cfg[\'train_max_side\'])\n            if len(self.cache) > self.cfg[\'cache_images\']: self.cache.popitem(last=False)\n        self.cache.move_to_end(key)\n        x, y = self.cache[key]\n        if crop:\n            h, w = x.shape[-2:]; size = min(crop, h, w)\n            top, left = random.randrange(h-size+1), random.randrange(w-size+1)\n            x, y = [v[:, top:top+size, left:left+size] for v in (x, y)]\n            if random.random() < .5: x, y = x.flip(-1), y.flip(-1)\n            if random.random() < .5: x, y = x.flip(-2), y.flip(-2)\n            turn = random.randrange(4); x, y = x.rot90(turn, (-2, -1)), y.rot90(turn, (-2, -1))\n        return x, y\n\n\ndef restoration_loss(pred, gt, cfg, target=None):\n    pred, gt = pred.float(), gt.float()\n    border = cfg[\'crop_border\']\n    if border:\n        pred, gt = [v[..., border:-border, border:-border] for v in (pred, gt)]\n        if target is not None: target = target[..., border:-border, border:-border]\n    edge = (F.l1_loss(pred[..., 1:, :]-pred[..., :-1, :], gt[..., 1:, :]-gt[..., :-1, :])\n            + F.l1_loss(pred[..., :, 1:]-pred[..., :, :-1], gt[..., :, 1:]-gt[..., :, :-1]))*.5\n    loss = F.mse_loss(pred, gt)+cfg[\'l1_weight\']*F.l1_loss(pred, gt)+cfg[\'edge_weight\']*edge\n    if target is not None: loss = loss+cfg[\'kd_weight\']*F.mse_loss(pred, target.float())\n    return loss\n\n\n@torch.no_grad()\ndef update_ema(ema, model, decay):\n    for target, source in zip(ema.parameters(), model.parameters()): target.lerp_(source, 1-decay)\n    for target, source in zip(ema.buffers(), model.buffers()): target.copy_(source)\n\n\ndef cpu_state(model):\n    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n\n\ndef rng_state():\n    return {\'python\': random.getstate(), \'numpy\': np.random.get_state(), \'torch\': torch.get_rng_state(),\n            \'cuda\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\n\ndef restore_rng(value):\n    random.setstate(value[\'python\']); np.random.set_state(value[\'numpy\']); torch.set_rng_state(value[\'torch\'])\n    if torch.cuda.is_available() and value[\'cuda\']: torch.cuda.set_rng_state_all(value[\'cuda\'])\n\n\ndef train_model(model, cfg, inputs, output, label, schedule, origin, teacher=None, mode=\'auto\'):\n    device = device_for(cfg); model.to(device)\n    best_path, last_path = output/f\'{label}_best.pth\', output/f\'{label}_last.pth\'\n    run_identity = identity(cfg, inputs)\n    if best_path.is_file():\n        saved = torch.load(best_path, map_location=\'cpu\', weights_only=False)\n        if saved[\'identity\'] != run_identity: raise ValueError(\'Trọng số không khớp run hiện tại.\')\n        if mode in (\'auto\', \'benchmark\'):\n            last = torch.load(last_path, map_location=\'cpu\', weights_only=False) if last_path.is_file() else None\n            if mode == \'benchmark\' or (last and last[\'completed\']):\n                model.load_state_dict(saved[\'params_ema\'], strict=True)\n                return saved.get(\'training\', {\'best_step\': saved[\'step\'], \'best_val_PSNR\': saved[\'best_val_PSNR\']})\n        if mode == \'train\': raise FileExistsError(f\'Đã có {best_path}; đổi RUN_NAME hoặc dùng resume.\')\n    elif mode in (\'resume\', \'benchmark\'):\n        raise FileNotFoundError(f\'Không có {best_path}.\')\n    if not inputs[\'splits\'][\'train\'] or not inputs[\'splits\'][\'val\']:\n        raise ValueError(\'Cần train/val riêng để cập nhật trọng số; SOTS cần TRAIN_ROOT ITS/OTS.\')\n    for stage in schedule:\n        if stage[\'steps\'] < 1 or stage[\'crop\'] < 0 or stage[\'accum\'] < 1:\n            raise ValueError(\'Schedule không hợp lệ.\')\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    if teacher is not None: teacher.to(device).eval().requires_grad_(False)\n    initial = quality(ema, inputs[\'splits\'][\'val\'], cfg, device)\n    best = metric_mean(initial, \'PSNR\'); best_step = step = 0\n    history = []; resume = None\n    if mode in (\'auto\', \'resume\') and last_path.is_file():\n        resume = torch.load(last_path, map_location=\'cpu\', weights_only=False)\n        if resume[\'identity\'] != run_identity: raise ValueError(\'Resume không khớp dữ liệu/cấu hình/mã.\')\n        model.load_state_dict(resume[\'params\']); ema.load_state_dict(resume[\'params_ema\'])\n        best, best_step, step, history = resume[\'best_val_PSNR\'], resume[\'best_step\'], resume[\'step\'], resume[\'history\']\n    started = time.perf_counter(); cache = PairCache(inputs[\'splits\'][\'train\'], cfg)\n    def artifact():\n        return {\'arch_version\': arch.ARCH_VERSION, \'model_spec\': model.spec, \'identity\': run_identity,\n                \'params_ema\': cpu_state(ema), \'step\': step, \'best_step\': best_step, \'best_val_PSNR\': best,\n                \'origin\': origin, \'training\': {\'best_step\': best_step, \'best_val_PSNR\': best, \'steps\': step}}\n    if not best_path.is_file():\n        atomic_save(artifact(), best_path); data.write_csv(output/f\'{label}_initial_val.csv\', initial)\n    for stage_id, stage in enumerate(schedule):\n        if resume and stage_id < resume[\'stage_id\']: continue\n        den_only = stage.get(\'den_only\', False)\n        for name, param in model.named_parameters():\n            param.requires_grad_(not den_only or name.startswith((\'trunk.restoration_network.DE.\', \'adapter.\')))\n        den, trunk = [], []\n        for name, param in model.named_parameters():\n            if param.requires_grad:\n                (den if name.startswith((\'trunk.restoration_network.DE.\', \'adapter.\')) else trunk).append(param)\n        groups = [{\'params\': den, \'lr\': stage[\'lr\'], \'initial_lr\': stage[\'lr\']}]\n        if trunk: groups.append({\'params\': trunk, \'lr\': stage.get(\'trunk_lr\', stage[\'lr\']),\n                                 \'initial_lr\': stage.get(\'trunk_lr\', stage[\'lr\'])})\n        opt = torch.optim.AdamW(groups, weight_decay=cfg[\'weight_decay\'])\n        local_start, stale = 0, 0\n        if resume and stage_id == resume[\'stage_id\']:\n            opt.load_state_dict(resume[\'optimizer\']); local_start, stale = resume[\'local_step\'], resume[\'stale\']\n            restore_rng(resume[\'rng\'])\n        for local_step in range(local_start+1, stage[\'steps\']+1):\n            model.train()\n            if origin != \'random\':\n                # Preserve inherited running statistics during small-domain adaptation.\n                for module in model.modules():\n                    if isinstance(module, nn.modules.batchnorm._BatchNorm): module.eval()\n            progress = local_step/stage[\'steps\']\n            warm = min(200, stage[\'steps\']//10)\n            factor = local_step/max(1, warm) if local_step <= warm else .02+.98*.5*(1+math.cos(math.pi*(local_step-warm)/max(1, stage[\'steps\']-warm)))\n            for group in opt.param_groups: group[\'lr\'] = group[\'initial_lr\']*factor\n            opt.zero_grad(set_to_none=True); loss_sum = 0.\n            for _ in range(stage[\'accum\']):\n                x, gt = cache.sample(stage[\'crop\']); x, gt = x[None].to(device), gt[None].to(device)\n                with torch.no_grad(): target = teacher(x) if teacher is not None else None\n                pred = model(x); loss = restoration_loss(pred, gt, cfg, target)\n                if not torch.isfinite(loss): raise FloatingPointError(f\'Loss NaN/Inf tại step {step+1}.\')\n                (loss/stage[\'accum\']).backward(); loss_sum += loss.item()/stage[\'accum\']\n            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1., error_if_nonfinite=True)\n            opt.step(); step += 1\n            update_ema(ema, model, min(cfg[\'ema_decay\'], (1+step)/(10+step)))\n            if step % cfg[\'log_every\'] == 0:\n                print(f\'{label}/{stage_id}: {local_step}/{stage["steps"]} loss={loss_sum:.6f}\', flush=True)\n            if local_step % cfg[\'val_every\'] == 0 or local_step == stage[\'steps\']:\n                rows = quality(ema, inputs[\'splits\'][\'val\'], cfg, device); score = metric_mean(rows, \'PSNR\')\n                if not math.isfinite(score): raise FloatingPointError(\'Validation PSNR không hữu hạn.\')\n                if score > best:\n                    best, best_step, stale = score, step, 0\n                    atomic_save(artifact(), best_path); data.write_csv(output/f\'{label}_best_val.csv\', rows)\n                else: stale += 1\n                history.append({\'step\': step, \'stage\': stage_id, \'local_step\': local_step, \'loss\': loss_sum,\n                                \'val_PSNR\': score, \'best_val_PSNR\': best, \'seconds\': time.perf_counter()-started})\n                data.write_csv(output/f\'{label}_history.csv\', history)\n                end_stage = local_step == stage[\'steps\'] or stale >= cfg[\'patience\']\n                atomic_save({**artifact(), \'params\': cpu_state(model), \'optimizer\': opt.state_dict(),\n                             \'stage_id\': stage_id, \'local_step\': stage[\'steps\'] if end_stage else local_step,\n                             \'stale\': stale, \'rng\': rng_state(), \'history\': history,\n                             \'completed\': end_stage and stage_id == len(schedule)-1}, last_path)\n                print(f\'{label}: validation={score:.4f}; best={best:.4f} tại step {best_step}\', flush=True)\n                if end_stage: break\n        # Each new stage starts at the validation optimum rather than an overfit iterate.\n        saved = torch.load(best_path, map_location=\'cpu\', weights_only=False)\n        model.load_state_dict(saved[\'params_ema\']); ema.load_state_dict(saved[\'params_ema\'])\n        resume = None\n    del ema, cache\n    if teacher is not None: teacher.cpu()\n    return {\'steps\': step, \'best_step\': best_step, \'best_val_PSNR\': best, \'seconds_this_session\': time.perf_counter()-started}\n\n\n@torch.inference_mode(False)\n@torch.no_grad()\ndef choose_runtime(model, cfg, inputs, output):\n    print(\'Runtime: bắt đầu kiểm tra Conv–BN fusion và đo trên các probe.\', flush=True)\n    device = device_for(cfg); model.to(device).eval()\n    if inputs[\'splits\'][\'val\']:\n        probes = [data.load_pair(r, cfg[\'eval_max_side\'])[0][None].to(device) for r in inputs[\'splits\'][\'val\'][:4]]\n        selection = \'validation\'\n    else:\n        # Synthetic probes only: test images and labels are not consulted for runtime selection.\n        gen = torch.Generator(device=device).manual_seed(cfg[\'seed\']+991)\n        probes = [torch.rand(1, 3, h, w, device=device, generator=gen) for h,w in ((256,256),(512,768))]\n        selection = \'synthetic\'\n    fused = copy.deepcopy(model).eval(); count = arch.fuse_conv_bn(fused)\n    errors = [float((model(x)-fused(x)).abs().max()) for x in probes]\n    print(f\'Runtime: {len(probes)} probe, {count} cặp fusion; đo eager...\', flush=True)\n    eager_ms = np.mean([latency(model, x, cfg)[\'latency_ms\'] for x in probes])\n    print(\'Runtime: đo bản fusion...\', flush=True)\n    fused_ms = np.mean([latency(fused, x, cfg)[\'latency_ms\'] for x in probes])\n    acceptable = max(errors) <= cfg[\'fusion_max_error\']\n    use = count > 0 and acceptable and fused_ms < eager_ms*.98\n    info = {\'execution\': \'conv_bn_fused\' if use else \'eager\', \'selection\': selection,\n            \'fused_pairs\': count, \'max_abs_error\': max(errors), \'eager_probe_ms\': float(eager_ms),\n            \'fused_probe_ms\': float(fused_ms), \'accepted\': bool(use)}\n    data.dump_json(output/\'runtime_selection.json\', info)\n    print(f\'Runtime: chọn {info["execution"]}; eager={eager_ms:.2f}ms, fused={fused_ms:.2f}ms.\', flush=True)\n    result = fused if use else model\n    model.cpu()\n    if not use: fused.cpu()\n    return result.cpu(), info\n\n\ndef evaluate_one(model, label, execution, cfg, inputs, output):\n    device = device_for(cfg); model.to(device).eval()\n    total = len(inputs[\'splits\'][\'test\'])\n    print(f\'Benchmark {label}: bắt đầu đánh giá {total} ảnh test ({execution}).\', flush=True)\n    rows = quality(model, inputs[\'splits\'][\'test\'], cfg, device, output/\'predictions\'/label, progress_label=label)\n    # Save quality before profiling/timing so a later failure does not erase it.\n    data.write_csv(output/f\'{label}_metrics.csv\', rows)\n    timing, flops = [], []\n    shape_cache = {}\n    started = time.perf_counter()\n    print(f\'{label}: bắt đầu đo FPS/GFLOPs; warmup={cfg["warmup"]}, \'\n          f\'repeats={cfg["timing_repeats"]}, shape_cache={cfg.get("cache_shape_benchmarks", False)}.\', flush=True)\n    for index, row in enumerate(inputs[\'splits\'][\'test\'], 1):\n        x, _ = data.load_pair(row, cfg[\'eval_max_side\']); probe = x[None].to(device)\n        shape = tuple(probe.shape)\n        cached = cfg.get(\'cache_shape_benchmarks\', False) and shape in shape_cache\n        if not cached:\n            measured = (latency(model, probe, cfg), profile_gflops(model, probe), row[\'id\'])\n            shape_cache[shape] = measured\n        measured = shape_cache[shape]\n        timing.append({\'id\': row[\'id\'], \'timing_source\': \'shape_cache\' if cached else \'measured\',\n                       \'representative_id\': measured[2], **measured[0]})\n        flops.append({\'id\': row[\'id\'], **measured[1]})\n        if index == 1 or index % 10 == 0 or index == total:\n            elapsed = time.perf_counter()-started\n            print(f\'{label}: FPS/GFLOPs {index}/{total} ảnh; {elapsed:.1f}s; \'\n                  f\'ETA ~{elapsed/index*(total-index)/60:.1f} phút.\', flush=True)\n    print(f\'{label}: đo probe 256×256 và ghi báo cáo.\', flush=True)\n    probe = torch.zeros(1, 3, 256, 256, device=device)\n    t256, f256 = latency(model, probe, cfg), profile_gflops(model, probe)\n    fps = 1000/metric_mean(timing, \'latency_ms\'); mse = metric_mean(rows, \'MSE\')\n    for quality_row, timing_row in zip(rows, timing):\n        quality_row.update(FPS=timing_row[\'FPS\'], FPS_over_MSE=data.efficiency(timing_row[\'FPS\'], quality_row[\'MSE\']))\n    data.write_csv(output/f\'{label}_metrics.csv\', rows)\n    data.write_csv(output/f\'{label}_timing.csv\', timing); data.write_csv(output/f\'{label}_gflops.csv\', flops)\n    data.dump_json(output/f\'{label}_256.json\', {\'timing\': t256, \'gflops\': f256})\n    result = {\'dataset\': cfg[\'dataset_name\'], \'model\': label, \'execution\': execution,\n              \'precision\': \'FP32\', \'batch_size\': 1, \'images\': len(rows), \'eval_max_side\': cfg[\'eval_max_side\'],\n              \'crop_border\': cfg[\'crop_border\'], \'PSNR\': metric_mean(rows, \'PSNR\'), \'SSIM\': metric_mean(rows, \'SSIM\'),\n              \'MSE\': mse, \'latency_ms\': metric_mean(timing, \'latency_ms\'), \'FPS\': fps, \'FPS_256\': t256[\'FPS\'],\n              \'FPS_over_MSE\': data.efficiency(fps, mse), \'params_M\': sum(p.numel() for p in model.parameters())/1e6,\n              \'GFLOPs_256_counted\': f256[\'counted_GFLOPs\'], \'GFLOPs_256_supplemental\': f256[\'supplemental_GFLOPs\'],\n              \'GFLOPs_256_estimated_total\': f256[\'estimated_total_GFLOPs\'],\n              \'GFLOPs_test_estimated_mean\': metric_mean(flops, \'estimated_total_GFLOPs\')}\n    model.cpu()\n    if device.type == \'cuda\': torch.cuda.empty_cache()\n    print(result, flush=True)\n    return result\n\n\ndef assess(reference, final, cfg):\n    gates = {\'quality\': final[\'PSNR\']-reference[\'PSNR\'] >= -cfg[\'psnr_tolerance_db\'],\n             \'speed\': final[\'FPS\']/reference[\'FPS\'] >= cfg[\'target_speedup\'],\n             \'speed_256\': final[\'FPS_256\']/reference[\'FPS_256\'] >= cfg[\'target_speedup_256\'],\n             \'efficiency\': final[\'FPS_over_MSE\'] >= reference[\'FPS_over_MSE\']}\n    return gates\n\n\ndef write_report(output, rows, metadata, cfg):\n    lines = [f\'# Phase 4 — WaveLite-Final / {cfg["dataset_name"]}\', \'\',\n             f\'Trạng thái: **{metadata["status"]}**.\', \'\',\n             \'| Model | Thực thi | PSNR | SSIM | MSE | FPS | FPS/MSE | FPS 256 | GFLOPs 256* | Params M |\',\n             \'|---|---|---:|---:|---:|---:|---:|---:|---:|---:|\']\n    for r in rows:\n        lines.append(f\'| {r["model"]} | {r["execution"]} | {r["PSNR"]:.4f} | {r["SSIM"]:.5f} | {r["MSE"]:.8f} | \'\n                     f\'{r["FPS"]:.2f} | {r["FPS_over_MSE"]:.2f} | {r["FPS_256"]:.2f} | \'\n                     f\'{r["GFLOPs_256_estimated_total"]:.3f} | {r["params_M"]:.3f} |\')\n    ref, final = rows[0], rows[-1]\n    compression = metadata.get(\'compression\', {})\n    architecture_note = (\'Chỉ tối ưu runtime: giữ nguyên kiến trúc và trọng số Reference; chưa có model nén.\'\n                         if metadata[\'origin\'][\'method\'] == \'runtime_only\' else\n                         f\'Kiến trúc: DEN {metadata["reference_spec"]["depth"]} → \'\n                         f\'{metadata["model_spec"]["depth"]} U-block/group; \'\n                         f\'giảm {100*compression.get("parameter_reduction", 0):.1f}% tham số.\')\n    # A concise, reviewable highlight uses the actual measured deltas only.\n    lines[4:4] = [f\'**PSNR {final["PSNR"]:.2f} dB · {final["FPS"]:.1f} FPS · \'\n                  f\'{final["GFLOPs_256_estimated_total"]:.2f} GFLOPs ở 256² · \'\n                  f\'FPS/MSE {final["FPS_over_MSE"]:,.0f}**\', \'\']\n    lines += [\'\', architecture_note, \'\', f\'ΔPSNR: {final["PSNR"]-ref["PSNR"]:+.4f} dB; tăng tốc: {final["FPS"]/ref["FPS"]:.3f}×; \'\n              f\'tỉ lệ FPS/MSE: {final["FPS_over_MSE"]/ref["FPS_over_MSE"]:.3f}×.\',\n              f\'Mục tiêu: giảm PSNR ≤ {cfg["psnr_tolerance_db"]:.2f} dB, FPS ≥ {cfg["target_speedup"]:.2f}× \'\n              f\'và FPS 256 ≥ {cfg["target_speedup_256"]:.2f}×. REVIEW nghĩa là chưa đạt đủ điều kiện.\', \'\',\n              f\'Kiến trúc thực đo: DWT → nhánh Mamba trên LL, giữ các dải cao → IWT → DEN {metadata["model_spec"]["depth"]} U-block/group \'\n              \'và một residual depthwise block/group \'\n              \'với ba nhánh FFT/attention → residual depthwise detail. Bản runtime giữ cấu trúc đầy đủ \'\n              \'khi chỉ có bộ benchmark; kiến trúc thực đo được lưu trong metadata.\',\n              f\'RGB [0,1], batch 1, FP32, TF32 tắt; max-side={cfg["eval_max_side"]} \'\n              f\'(0 nghĩa là kích thước gốc), crop border={cfg["crop_border"]}. Không TTA/tiling.\',\n              \'PSNR/SSIM/MSE là trung bình đều theo ảnh. FPS = 1000 / latency trung bình theo ảnh. \'\n              \'FPS/MSE = FPS / MSE trung bình; không lấy trung bình tỉ số và không suy MSE từ PSNR trung bình.\',\n              \'FPS đo forward gồm pad/crop, đã warm-up; không gồm đọc ảnh, H2D/D2H và tính metric. \'\n              \'Tất cả model đo lại trong cùng phiên GPU.\',\n              \'*GFLOPs: 1 MAC=2 FLOPs; Conv/Linear và SS2D projections counted; scan/FFT/Haar ước tính bổ sung. \'\n              \'Normalization, activation, pooling và elementwise chưa được đếm đầy đủ. CSV giữ từng thành phần.\',\n              \'Test chỉ dùng lập bảng cuối; validation quyết định trọng số và runtime. \'\n              \'FPS/MSE là metric phụ thuộc thiết bị và protocol, không thay thế điều kiện PSNR riêng.\',\n              \'Thông tin tái lập và nguồn trọng số: metadata.json, config.json, inputs.json và SHA256 mã/dữ liệu.\']\n    if cfg[\'dataset_name\'] == \'o-hazy\':\n        lines += [\'O-HAZY: test 41–45 đã được xem ở các phase trước; đây là đánh giá trong quá trình phát triển. \'\n                  \'Validation có thể trùng scene từng dùng để huấn luyện mô hình tham chiếu, nên không phải holdout độc lập.\']\n    if cfg.get(\'cache_shape_benchmarks\'):\n        lines += [\'Đánh giá chất lượng trên toàn bộ test. Latency/GFLOPs đo một ảnh đại diện cho mỗi \'\n                  \'kích thước đầu vào rồi dùng lại cho các ảnh cùng kích thước; CSV lưu representative_id.\']\n    (output/\'protocol_report.md\').write_text(\'\\n\'.join(lines)+\'\\n\', encoding=\'utf-8\')\n    if cfg.get(\'preset\') == \'quick\':\n        lines = lines[:6] + lines[6:8+len(rows)] + [\n            \'\', f\'**Tăng tốc {final["FPS"]/ref["FPS"]:.2f}× · ΔPSNR {final["PSNR"]-ref["PSNR"]:+.3f} dB · \'\n            f\'FPS/MSE {final["FPS_over_MSE"]/ref["FPS_over_MSE"]:.2f}×.**\',\n            architecture_note,\n            f\'Test: {final["images"]} ảnh; FP32, batch 1; max-side {cfg["eval_max_side"]}; \'\n            f\'crop border {cfg["crop_border"]}. FPS/MSE = FPS / MSE RGB [0,1] trung bình theo ảnh.\',\n            \'GFLOPs gồm counted + ước tính scan/FFT/Haar; chưa đếm đầy đủ normalization/elementwise. \'\n            \'FPS không gồm I/O/transfer. REVIEW nghĩa là chưa đạt đầy đủ mục tiêu.\',\n            (\'Latency/GFLOPs được dùng lại theo kích thước ảnh; chất lượng đo toàn bộ test.\'\n             if cfg.get(\'cache_shape_benchmarks\') else \'Latency đo từng ảnh test.\'),\n            \'Chi tiết: [protocol](protocol_report.md), [metadata](metadata.json), [CSV](summary.csv).\']\n    (output/\'final_report.md\').write_text(\'\\n\'.join(lines)+\'\\n\', encoding=\'utf-8\')\n    write_html_report(output, rows, metadata, cfg)\n    print(\'\\n\'.join(lines), flush=True)\n\n\ndef write_html_report(output, rows, metadata, cfg):\n    ref, final = rows[0], rows[-1]\n    escape = lambda value: html.escape(str(value))\n    cards = [(\'PSNR\', f\'{final["PSNR"]:.2f}\', \'dB\'), (\'Tốc độ\', f\'{final["FPS"]:.1f}\', \'FPS\'),\n             (\'Hiệu quả\', f\'{final["FPS_over_MSE"]:,.0f}\', \'FPS/MSE\'),\n             (\'Độ phức tạp\', f\'{final["GFLOPs_256_estimated_total"]:.2f}\', \'GFLOPs / 256², ước tính\')]\n    body = \'\'.join(f\'<article><small>{escape(title)}</small><strong>{value}</strong><span>{unit}</span></article>\'\n                   for title,value,unit in cards)\n    table = \'\'.join(\'<tr\'+(\' class="selected"\' if i == len(rows)-1 else \'\')+\'>\'+\'\'.join(\n        f\'<td>{escape(value)}</td>\' for value in (r[\'model\'],r[\'execution\'],f\'{r["PSNR"]:.3f}\',\n        f\'{r["SSIM"]:.4f}\',f\'{r["MSE"]:.6f}\',f\'{r["FPS"]:.1f}\',f\'{r["FPS_over_MSE"]:,.0f}\',\n        f\'{r["GFLOPs_256_estimated_total"]:.2f}\'))+\'</tr>\' for i,r in enumerate(rows))\n    document = \'\'\'<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width">\n<title>Phase 4 — Kết quả khử sương</title><style>\nbody{margin:0;background:#f3f6fb;color:#162238;font:16px/1.6 system-ui,sans-serif}main{max-width:1100px;margin:48px auto;padding:28px}\nh1{font-size:36px;margin:4px 0}small,.muted{color:#61718c}.badge{padding:6px 14px;border-radius:20px;background:#e5eaf2;font-weight:700}\n.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:16px;margin:28px 0}article{background:white;padding:22px;border-radius:18px;box-shadow:0 5px 25px #182c4810}\narticle strong{display:block;font-size:38px;letter-spacing:-1px}article span{font-size:13px;color:#61718c}.delta{padding:18px 24px;border-left:4px solid #3978ee;background:#e9f0ff;border-radius:8px}\n.table{overflow:auto;margin:24px 0}table{border-collapse:collapse;background:white;width:100%;font-size:14px}td,th{padding:14px;text-align:right;border-bottom:1px solid #e8edf4;white-space:nowrap}td:first-child,th:first-child{text-align:left}.selected{background:#edf6ff;font-weight:700}footer{font-size:13px;color:#61718c}a{color:#2866c7}\n</style><main>\'\'\'\n    document += f\'<small>KẾT QUẢ THỰC NGHIỆM · PHASE 4</small><h1>{escape(cfg["dataset_name"].upper())}</h1>\'\n    document += f\'<span class="badge">{escape(metadata["status"])}</span><div class="cards">{body}</div>\'\n    if metadata[\'origin\'][\'method\'] == \'runtime_only\':\n        document += \'<p>Chỉ tối ưu runtime: giữ nguyên kiến trúc và trọng số Reference; chưa có model nén.</p>\'\n    document += f\'<div class="delta">Tăng tốc <b>{final["FPS"]/ref["FPS"]:.2f}×</b> · \'\n    document += f\'ΔPSNR <b>{final["PSNR"]-ref["PSNR"]:+.3f} dB</b> · \'\n    document += f\'FPS/MSE <b>{final["FPS_over_MSE"]/ref["FPS_over_MSE"]:.2f}×</b></div>\'\n    document += \'<div class="table"><table><tr><th>Model</th><th>Runtime</th><th>PSNR</th><th>SSIM</th><th>MSE</th><th>FPS</th><th>FPS/MSE</th><th>GFLOPs 256*</th></tr>\'+table+\'</table></div>\'\n    document += f\'<footer>{final["images"]} ảnh test · FP32 · batch 1 · max-side {cfg["eval_max_side"]} · \'\n    document += f\'crop border {cfg["crop_border"]}. FPS/MSE dùng MSE RGB [0,1] trung bình theo ảnh. \'\n    document += \'FPS loại I/O/transfer. *GFLOPs là ước tính một phần. REVIEW: chưa đạt đủ điều kiện PSNR/FPS.\'\n    if cfg.get(\'cache_shape_benchmarks\'): document += \' Latency/GFLOPs dùng chung theo shape; metric chất lượng đo mọi ảnh.\'\n    document += \'<p><a href="protocol_report.md">Protocol</a> · <a href="metadata.json">Metadata</a> · <a href="summary.csv">Số liệu CSV</a></p></footer></main></html>\'\n    (output/\'report.html\').write_text(document, encoding=\'utf-8\')\n\n\ndef run(cfg, mode=\'auto\'):\n    configure(cfg); device_for(cfg)\n    output = Path(cfg[\'output_root\']); output.mkdir(parents=True, exist_ok=True)\n    inputs = json.loads((Path(cfg[\'work_root\'])/\'inputs.json\').read_text(encoding=\'utf-8\')); validate(cfg, inputs)\n    require_adaptation(cfg, inputs)\n    scratch = cfg[\'dataset_name\'] == \'i-haze\'\n    if scratch:\n        if inputs[\'evaluation_only\']: raise ValueError(\'I-HAZE cần train/val để khởi tạo ngẫu nhiên.\')\n        reference = arch.reference_model(cfg[\'repo_root\'], depth=cfg[\'reference_depth\'])\n        reference_training = train_model(reference, cfg, inputs, output, \'reference\', cfg[\'reference_schedule\'], \'random\', mode=mode)\n        # Independently random initialization: final I-HAZE receives no reference weights or KD.\n        # Keep the final initialization identical whether reference training just\n        # ran or was restored by auto/resume in a new process.\n        configure({**cfg, \'seed\': cfg[\'seed\']+1000})\n        final = arch.skeleton(cfg[\'repo_root\'], {\'kind\': \'final\', \'depth\': 1, \'feature_adapter\': True,\n                                                \'detail_width\': cfg[\'detail_width\']})\n        # A pipeline interrupted during reference training has no final artifact\n        # yet; finishing that reference must still allow the final stage to start.\n        final_mode = \'auto\' if mode == \'resume\' and not (output/\'final_best.pth\').is_file() else mode\n        training = train_model(final, cfg, inputs, output, \'final\', cfg[\'scratch_schedule\'], \'random\', mode=final_mode)\n        origin = {\'method\': \'random\', \'reference_method\': \'random_independent\', \'source\': None,\n                  \'reference_seed\': cfg[\'seed\'], \'final_seed\': cfg[\'seed\']+1000}\n    else:\n        reference = arch.reference_model(cfg[\'repo_root\'], inputs[\'source\'])\n        reference_training = None\n        if inputs[\'evaluation_only\'] or cfg[\'dataset_name\'] in cfg.get(\'runtime_only_datasets\', []):\n            # No adaptation split: avoid irreversible pruning without recovery supervision.\n            final = copy.deepcopy(reference)\n            training = {\'mode\': \'runtime_only\', \'steps\': 0}\n            origin = {\'method\': \'runtime_only\', \'source\': inputs[\'source\']}\n        else:\n            start = adaptation_start(cfg, inputs, reference)\n            final, selected = arch.final_from_reference(start, cfg[\'den_blocks\'], cfg[\'detail_width\'])\n            if cfg.get(\'require_compression\') and not compression_info(reference, final)[\'compressed\']:\n                raise ValueError(\'Model cuối chưa giảm tham số so với Reference; dừng trước train.\')\n            print(\'Before recovery training:\', compression_info(reference, final), flush=True)\n            origin = {\'method\': \'inherited\', \'source\': inputs[\'source\'], \'student_source\': inputs[\'student_source\'],\n                      \'selected_blocks\': selected,\n                      \'initialization_source\': inputs[\'student_source\'] if start is not reference else inputs[\'source\'],\n                      \'transfer\': arch.transfer_summary(final)}\n            print(\'Checkpoint transfer:\', origin[\'transfer\'], \'selected blocks:\', selected, flush=True)\n            training = train_model(final, cfg, inputs, output, \'final\', cfg[\'adapt_schedule\'], \'inherited\',\n                                   teacher=reference, mode=mode)\n    compression = compression_info(reference, final)\n    if cfg.get(\'require_compression\') and not compression[\'compressed\']:\n        raise ValueError(\'Model cuối không có kiến trúc nhẹ hơn Reference; không xuất bản sao thành WaveLite.\')\n    print(\'Architecture check:\', compression, flush=True)\n    reference.cpu(); final.cpu()\n    deploy, runtime = choose_runtime(copy.deepcopy(final), cfg, inputs, output)\n    # Runtime selection is sealed before any test quality is computed.\n    rows = []\n    jobs = [(reference, \'Reference\', \'eager\')]\n    runtime_only = origin[\'method\'] == \'runtime_only\'\n    if not (runtime_only and cfg.get(\'skip_duplicate_eager\', False)):\n        jobs.append((final, \'Reference_copy\' if runtime_only else \'WaveLite_eager\', \'eager\'))\n    jobs.append((deploy, \'Reference_runtime\' if runtime_only else \'WaveLite_deploy\', runtime[\'execution\']))\n    for model, label, execution in jobs:\n        rows.append(evaluate_one(model, label, execution, cfg, inputs, output))\n        data.write_csv(output/\'summary_partial.csv\', rows)\n    gates = assess(rows[0], rows[-1], cfg)\n    if cfg.get(\'require_compression\'): gates[\'architecture_compression\'] = compression[\'compressed\']\n    gates[\'deployment_quality\'] = abs(rows[-1][\'PSNR\']-rows[-2][\'PSNR\']) <= cfg[\'deploy_psnr_tolerance_db\']\n    metadata = {\'status\': \'PASS\' if all(gates.values()) else \'REVIEW\', \'gates\': gates,\n                \'origin\': origin, \'architecture_version\': arch.ARCH_VERSION, \'model_spec\': final.spec,\n                \'reference_spec\': reference.spec, \'training\': training, \'reference_training\': reference_training,\n                \'runtime\': runtime, \'compression\': compression, \'identity\': identity(cfg, inputs),\n                \'gpu\': torch.cuda.get_device_name() if torch.cuda.is_available() else \'CPU\',\n                \'torch\': torch.__version__, \'cuda\': torch.version.cuda,\n                \'FPS_MSE_definition\': \'FPS / arithmetic mean per-image RGB MSE on [0,1], same crop as PSNR\',\n                \'metric_scope\': \'same-session FP32, batch=1, raw input pad/crop included; I/O and transfer excluded\'}\n    data.write_csv(output/\'summary.csv\', rows); data.dump_json(output/\'metadata.json\', metadata)\n    data.dump_json(output/\'config.json\', cfg)\n    if (output/\'final_best.pth\').is_file(): metadata[\'final_sha256\'] = data.sha256(output/\'final_best.pth\')\n\n    # Self-contained eager export: reconstruction needs the architecture files,\n    # not the original source weights. Fusion remains an inference-only transform.\n    atomic_save({\'arch_version\': arch.ARCH_VERSION, \'model_spec\': final.spec,\n                 \'params_ema\': cpu_state(final), \'identity\': identity(cfg, inputs),\n                 \'origin\': origin, \'runtime\': runtime}, output/\'model_final.pth\')\n    metadata[\'export_sha256\'] = data.sha256(output/\'model_final.pth\')\n    data.dump_json(output/\'metadata.json\', metadata); write_report(output, rows, metadata, cfg)\n\n\nif __name__ == \'__main__\':\n    cfg = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\')); action = sys.argv[2]\n    if action == \'prepare\': prepare(cfg)\n    elif action in (\'auto\', \'train\', \'resume\', \'benchmark\'): run(cfg, action)\n    else: raise ValueError(\'Chọn prepare, auto, train, resume hoặc benchmark.\')\n', 'checkpoint_setup.py': '"""Shared checkpoint bootstrap. Does not modify model/training sources or artifacts."""\nfrom pathlib import Path\nimport json\nimport subprocess\nimport sys\nimport zipfile\n\n\nWEIGHT_NAMES = {\'o-hazy\': \'Ohaze_1200_1600_27.22.pth\',\n                \'reside6k\': \'reside6k_32.15.pth\',\n                \'sots-its\': \'reside6k_32.15.pth\', \'sots-ots\': \'reside6k_32.15.pth\'}\n\n\ndef download_archive(target):\n    try:\n        import gdown\n    except ImportError:\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'gdown\'], check=True)\n        import gdown\n    result = gdown.download(id=\'1Vm-1mtMSc9YIUICq4nWKu-4AHCtYnPpF\', output=str(target),\n                            quiet=False, use_cookies=False, resume=True)\n    if result is None:\n        raise RuntimeError(\'Google Drive không trả về ZIP trọng số. Kiểm tra quota hoặc đặt CHECKPOINT_ZIP local.\')\n\n\ndef find_weight(roots, filename):\n    for root in roots:\n        root = Path(root).expanduser()\n        if root.is_dir():\n            found = sorted(root.rglob(filename))\n            found = [p for p in found if p.is_file() and p.stat().st_size]\n            if found:\n                return found[0].resolve()\n    return None\n\n\ndef ensure_shared_weights(configs, cache_root, download=download_archive):\n    """Populate shared cache once; keep failures per dataset and leave explicit paths alone."""\n    cache = Path(cache_root).expanduser().resolve()\n    cache.mkdir(parents=True, exist_ok=True)\n    missing, errors = [], {}\n    for cfg in configs:\n        name = cfg[\'dataset_name\']\n        if name not in WEIGHT_NAMES:\n            continue  # I-HAZE never needs pretrained weights.\n        roots = cfg.setdefault(\'checkpoint_search_roots\', [])\n        if str(cache) not in roots:\n            roots.append(str(cache))\n        if cfg.get(\'repo_root\') and cfg[\'repo_root\'] not in roots:\n            roots.append(cfg[\'repo_root\'])\n        if cfg.get(\'checkpoint\') or (name != \'o-hazy\' and cfg.get(\'reside_checkpoint\')):\n            continue  # Explicit path validation remains in the runner.\n        search = [cfg.get(\'project_root\', \'\'), cfg.get(\'repo_root\', \'\'),\n                  Path(cfg[\'work_root\'])/\'weights\', *roots]\n        if not find_weight([p for p in search if p], WEIGHT_NAMES[name]):\n            missing.append(cfg)\n    if not missing:\n        return errors\n    # A supplied ZIP works even when network downloads are disabled.\n    provided = next((c[\'checkpoint_zip\'] for c in missing if c.get(\'checkpoint_zip\')), \'\')\n    downloadable = [c for c in missing if c.get(\'allow_weight_download\')]\n    if not provided and not downloadable:\n        return {c[\'dataset_name\']: \'Thiếu checkpoint và ALLOW_WEIGHT_DOWNLOAD=False.\' for c in missing}\n    archive = Path(provided).expanduser() if provided else cache/\'WDMamba_ckpts.zip\'\n    try:\n        if not provided and not zipfile.is_zipfile(archive):\n            partial = archive.with_suffix(\'.zip.partial\')\n            print(\'Đang tải bộ checkpoint dùng chung cho SOTS và O-HAZY...\', flush=True)\n            download(partial)\n            if not zipfile.is_zipfile(partial):\n                raise ValueError(\'File tải về không phải ZIP trọng số hợp lệ.\')\n            partial.replace(archive)\n        target = cache/\'unzipped\'; target.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(archive) as z:\n            if z.testzip() is not None:\n                raise ValueError(\'ZIP trọng số bị lỗi CRC.\')\n            for item in z.infolist():\n                if not (target/item.filename).resolve().is_relative_to(target):\n                    raise ValueError(\'ZIP chứa đường dẫn ngoài thư mục giải nén.\')\n            z.extractall(target)\n        for cfg in missing:\n            name = cfg[\'dataset_name\']\n            if not find_weight([cache], WEIGHT_NAMES[name]):\n                errors[name] = f\'ZIP không chứa {WEIGHT_NAMES[name]}.\'\n    except Exception as exc:\n        errors = {c[\'dataset_name\']: str(exc) for c in missing}\n    for name, message in errors.items():\n        print(f\'{name}: chuẩn bị checkpoint lỗi: {message}\', flush=True)\n    return errors\n\n\nif __name__ == \'__main__\':\n    path = Path(sys.argv[1])\n    bundle = json.loads(path.read_text(encoding=\'utf-8\'))\n    bundle[\'errors\'] = ensure_shared_weights(bundle[\'configs\'], bundle[\'cache_root\'])\n    path.write_text(json.dumps(bundle, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n'}

for name, source in MODULE_SOURCES.items():
    ast.parse(source)
    (RUN_ROOT/name).write_text(source, encoding='utf-8')
if str(RUN_ROOT) not in sys.path: sys.path.insert(0, str(RUN_ROOT))
batch_ns = {'__name__': 'fresh_batch'}
exec(compile(MODULE_SOURCES['phase4_batch.py'], 'phase4_batch.py', 'exec'), batch_ns)
make_configs, prepare_jobs, run_jobs, collect_reports = [batch_ns[k] for k in ('make_configs','prepare_jobs','run_jobs','collect_reports')]

if DATASET_NAME == 'all':
    CONFIGS = make_configs(CONFIG, DATASET_SPECS, RUN_ROOT, DATA_ROOT_OVERRIDES, TRAIN_ROOTS,
                           CHECKPOINTS, {'o-hazy': STUDENT_CHECKPOINT})
else:
    CONFIGS = [CONFIG]

for cfg in CONFIGS:
    if cfg['dataset_name'] == 'i-haze' and PRESET == 'quick':
        cfg['val_every'] = 200
        cfg['patience'] = 30  # > all validation checks in either quick stage

# Resolve the shared archive once before starting all jobs. This prevents SOTS
# Indoor/Outdoor from independently downloading the same checkpoint ZIP.
checkpoint_ns = {'__name__': 'checkpoint_setup_bootstrap'}
exec(compile(MODULE_SOURCES['checkpoint_setup.py'], 'checkpoint_setup.py', 'exec'), checkpoint_ns)
checkpoint_errors = checkpoint_ns['ensure_shared_weights'](
    CONFIGS, SHARED_WEIGHT_ROOT)
if checkpoint_errors:
    print('Checkpoint status:', checkpoint_errors, flush=True)


if checkpoint_errors:
    raise RuntimeError('Chưa đủ checkpoint, chưa bắt đầu train: ' + json.dumps(checkpoint_errors, ensure_ascii=False))
print('Đã chuẩn bị checkpoint cho toàn bộ dataset được chọn.', flush=True)


## 3. Môi trường CUDA/Mamba

Tự cài môi trường nếu chưa có; tái dùng môi trường hợp lệ không tái dùng kết quả model cũ.

In [ ]:

check = r"""
import torch
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
import cv2, skimage, timm, einops, pytorch_wavelets
assert torch.cuda.is_available()
u = torch.zeros(1,4,16,device='cuda')
y = selective_scan_fn(u, torch.ones_like(u), -torch.ones(4,1,device='cuda'),
                      torch.ones(1,1,1,16,device='cuda'), torch.ones(1,1,1,16,device='cuda'), delta_softplus=True)
assert torch.isfinite(y).all()
print('CUDA/Mamba PASS:', torch.__version__, torch.version.cuda, torch.cuda.get_device_name())
"""
candidates = [Path(PYTHON_OVERRIDE)] if PYTHON_OVERRIDE else [
    Path('/content/wdmamba_phase4_env311/bin/python'),
    Path('/content/wdmamba_o_hazy_env311/bin/python'), Path(sys.executable)]
PYTHON = None
for candidate in candidates:
    if candidate.is_file():
        probe = subprocess.run([str(candidate), '-c', check], capture_output=True, text=True)
        if probe.returncode == 0:
            PYTHON = candidate; print(probe.stdout); break
        print('Môi trường chưa phù hợp:', candidate, probe.stderr[-1200:])
if PYTHON is None:
    if PYTHON_OVERRIDE: raise RuntimeError('PYTHON_OVERRIDE không vượt qua kiểm tra CUDA/Mamba.')
    command(['nvidia-smi']); command(['nvcc','--version'])
    command([sys.executable,'-m','pip','install','-q','uv'])
    command([sys.executable,'-m','uv','python','install','3.11'])
    env_root = Path('/content/wdmamba_phase4_env311')
    PYTHON = env_root/'bin/python'
    if not PYTHON.is_file(): command([sys.executable,'-m','uv','venv','--python','3.11',env_root])
    command([sys.executable,'-m','uv','pip','install','--python',PYTHON,
             '--index-url','https://download.pytorch.org/whl/cu128','torch==2.8.0','torchvision==0.23.0'])
    constraints = env_root/'constraints.txt'
    constraints.write_text('torch==2.8.0\ntorchvision==0.23.0\nnumpy<2\nsetuptools<81\n', encoding='utf-8')
    command([sys.executable,'-m','uv','pip','install','--python',PYTHON,'--constraint',constraints,
             'numpy<2','opencv-python-headless<4.12','scipy','scikit-image','einops','timm==1.0.15',
             'pytorch-wavelets','PyWavelets','setuptools<81','packaging','ninja','wheel','transformers','gdown','tqdm'])
    command([sys.executable,'-m','uv','pip','install','--python',PYTHON,'--constraint',constraints,
             '--no-build-isolation','causal-conv1d','mamba-ssm'])
    command([PYTHON,'-c',check])
(OUTPUT_ROOT/'environment.txt').write_text(subprocess.check_output(
    [str(PYTHON),'-m','pip','freeze'], text=True) if subprocess.run(
    [str(PYTHON),'-m','pip','--version'], capture_output=True).returncode == 0 else subprocess.check_output(
    [sys.executable,'-m','uv','pip','freeze','--python',str(PYTHON)], text=True), encoding='utf-8')


## 4. Kiểm tra toàn bộ job trước khi train

Kiểm tra split, nguồn trọng số, nạp strict và forward tensor giả trên GPU. Cả bốn bộ phải sẵn sàng.
Test không được dùng chọn model hoặc runtime.

In [ ]:
ARCH_PREFLIGHT_SOURCE = "\nimport json, sys\nfrom pathlib import Path\nimport torch\ncfg_path = Path(sys.argv[1]); sys.path.insert(0, str(cfg_path.parent))\nimport phase4_arch as arch\nimport phase4_runner as runner\ncfg = json.loads(cfg_path.read_text(encoding='utf-8'))\ninputs = json.loads(cfg_path.with_name('inputs.json').read_text(encoding='utf-8'))\nrunner.configure(cfg)\nref = arch.reference_model(cfg['repo_root'], inputs['source'] or None, depth=cfg['reference_depth'])\nmodels = [ref]\nif cfg['dataset_name'] == 'i-haze':\n    models.append(arch.skeleton(cfg['repo_root'], {'kind':'final','depth':1,'feature_adapter':True,'detail_width':cfg['detail_width']}))\nelif not (inputs['evaluation_only'] or cfg['dataset_name'] in cfg.get('runtime_only_datasets', [])):\n    start = runner.adaptation_start(cfg, inputs, ref)\n    models.append(arch.final_from_reference(start, cfg['den_blocks'], cfg['detail_width'])[0])\nrunner.require_adaptation(cfg, inputs)\nif cfg.get('require_compression') and (len(models) < 2 or not runner.compression_info(ref, models[-1])['compressed']):\n    raise RuntimeError('Preflight: model cuối chưa có kiến trúc nhẹ hơn Reference.')\nprint('Architecture preflight:', [m.spec for m in models], flush=True)\nif len(models) > 1: print('Compression preflight:', runner.compression_info(ref, models[-1]), flush=True)\nif cfg['dataset_name'] != 'i-haze' and len(models) > 1:\n    print('Checkpoint transfer preflight:', arch.transfer_summary(models[-1]), flush=True)\nprobe = torch.rand(1, 3, 256, 256, device=cfg['device'])\nfor model in models:\n    model.to(cfg['device']).eval()\n    with torch.no_grad(): result = model(probe)\n    if result.shape != probe.shape or not torch.isfinite(result).all():\n        raise RuntimeError('Forward smoke test không hợp lệ.')\n    model.cpu()\nprint('PREFLIGHT PASS:', cfg['dataset_name'], 'checkpoint strict + forward GPU', flush=True)\n"
ARCH_PREFLIGHT_PATH = RUN_ROOT/'fresh_preflight.py'
ARCH_PREFLIGHT_PATH.write_text(ARCH_PREFLIGHT_SOURCE, encoding='utf-8')

def invoke_job(cfg, action):
    if action == 'prepare' and cfg['dataset_name'] in checkpoint_errors:
        raise RuntimeError(checkpoint_errors[cfg['dataset_name']])
    work = Path(cfg['work_root']); work.mkdir(parents=True, exist_ok=True)
    Path(cfg['output_root']).mkdir(parents=True, exist_ok=True)
    for name, source in MODULE_SOURCES.items():
        (work/name).write_text(source, encoding='utf-8')
    config_path = work/'config.json'
    config_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding='utf-8')
    command([PYTHON,'-u',work/'phase4_runner.py',config_path,action], cwd=REPO_ROOT,
            log=cfg['dataset_name']+'_'+action+'.log')
    if action == 'prepare':
        command([PYTHON, '-u', ARCH_PREFLIGHT_PATH, config_path], cwd=work,
                log=cfg['dataset_name']+'_preflight.log')
def run_phase(action):
    CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
    command([PYTHON,'-u',RUN_ROOT/'phase4_runner.py',CONFIG_PATH,action], cwd=REPO_ROOT, log=action+'.log')
if RUN_MODE not in ('auto','train','resume','benchmark'): raise ValueError('RUN_MODE không hợp lệ.')
if DATASET_NAME == 'all':
    SUITE_STATES = prepare_jobs(CONFIGS, invoke_job, OUTPUT_ROOT)
    print('Chuẩn bị:', SUITE_STATES)
    failed = [s for s in SUITE_STATES if s['status'] != 'ready']
    if failed:
        collect_reports(CONFIGS, SUITE_STATES, OUTPUT_ROOT)
        raise RuntimeError('Chưa đủ dataset sẵn sàng, chưa bắt đầu train. Xem log và suite_status.json: ' +
                           ', '.join(s['dataset'] for s in failed))
    print(f'Sẵn sàng {sum(s["status"] == "ready" for s in SUITE_STATES)}/{len(CONFIGS)} dataset.', flush=True)
else:
    if checkpoint_errors: raise RuntimeError(str(checkpoint_errors))
    invoke_job(CONFIG, 'prepare')
    inputs = json.loads((RUN_ROOT/'inputs.json').read_text(encoding='utf-8'))
    print('Split:', {k:len(v) for k,v in inputs['splits'].items()})
    print('Nguồn tham chiếu:', inputs['source'] or 'khởi tạo ngẫu nhiên I-HAZE')


## 5. Train, tối ưu runtime và đánh giá

Chạy đủ bốn bộ theo thứ tự SOTS Indoor → SOTS Outdoor → O-HAZY → I-HAZE.
Lỗi một job được lưu, các job khác tiếp tục; cuối lượt sẽ báo chưa hoàn tất nếu thiếu bộ nào.

In [ ]:

if DATASET_NAME == 'all':
    run_jobs(CONFIGS, SUITE_STATES, invoke_job, RUN_MODE, OUTPUT_ROOT)
    incomplete = [s['dataset'] for s in SUITE_STATES if s['status'] != 'completed']
    if incomplete:
        raise RuntimeError('Lượt all chưa hoàn tất: ' + ', '.join(incomplete)
                           + '. Xem suite_status.json và log; các kết quả đã xong vẫn được giữ.')
else:
    run_phase(RUN_MODE)


## 6. Báo cáo và metric

**FPS/MSE = FPS / MSE trung bình theo ảnh**, RGB [0,1], cùng crop border với PSNR. Không suy MSE từ
PSNR trung bình; không lấy trung bình các tỉ số. FPS đo batch 1, FP32, TF32 tắt, gồm pad/crop và loại
I/O/transfer/metrics. Đây là metric phụ thuộc GPU, không thay thế ngưỡng giảm PSNR riêng.

GFLOPs tại 256×256 và kích thước test được tách counted/supplemental. Model tham chiếu, eager và deploy
đều đo lại trong cùng runtime. `PASS` chỉ xuất khi đủ mọi điều kiện; số đo chưa đạt vẫn giữ nguyên.

In [ ]:

from IPython.display import Markdown, display
import csv
report_name = 'final_report_all.md' if DATASET_NAME == 'all' else 'final_report.md'
summary_name = 'summary_all.csv' if DATASET_NAME == 'all' else 'summary.csv'
display(Markdown((OUTPUT_ROOT/report_name).read_text(encoding='utf-8')))
print('Report trình bày:', OUTPUT_ROOT/('report_all.html' if DATASET_NAME == 'all' else 'report.html'))
with (OUTPUT_ROOT/summary_name).open(encoding='utf-8') as f:
    for row in csv.DictReader(f):
        print(row['dataset'], row['model'], 'MSE=',row['MSE'], 'FPS/MSE=',row['FPS_over_MSE'])


## 7. Xuất artifact

`model_final.pth` là bản eager tự chứa, kèm spec và nguồn gốc. Dựng lại bằng
`phase4_arch.skeleton(repo_root, saved['model_spec'])`, nạp `saved['params_ema']` strict=True rồi eval.
Không cần nguồn checkpoint ban đầu để dựng model đã xuất. Fusion chỉ thực hiện sau khi nạp eager.
Metadata/config/manifest lưu nguồn khởi tạo thực tế; report chính tập trung kiến trúc và kết quả.
O-HAZY giữ test 41–45, đã được xem ở các phase trước; không phải holdout mới.

In [ ]:

import shutil
for cfg in CONFIGS:
    work, destination = Path(cfg['work_root']), Path(cfg['output_root'])
    destination.mkdir(parents=True, exist_ok=True)
    for src in [work/name for name in MODULE_SOURCES] + [work/'config.json',work/'inputs.json',work/'split_manifest.csv']:
        if src.is_file(): shutil.copy2(src, destination/src.name)
    if (OUTPUT_ROOT/'environment.txt').is_file() and destination != OUTPUT_ROOT:
        shutil.copy2(OUTPUT_ROOT/'environment.txt', destination/'environment.txt')
archive_root = RUN_ROOT if DATASET_NAME == 'all' else OUTPUT_ROOT
archive = shutil.make_archive(str(RUN_ROOT.parent/(RUN_NAME+'_phase4_'+DATASET_NAME+'_results')), 'zip', archive_root)
print('Kết quả:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass
